In [171]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import re
from pandas import read_csv

In [172]:
def webscrab(plaintext):
    # Removed the '^' anchor so it searches globally across the continuous string
    patternn = r"(\d{4}-\d{2}-\d{2})\s*(\d{2}:\d{2}(?: \(\d{2}:\d{2}\))?).*?(Mon|Tue|Wed|Thu|Fri|Sat|Sun)\s*(Home|Away)"

    # Scan the entire text block and extract all occurrences instantly
    matchess = re.findall(patternn, plaintext)

    # Load the matches directly into a clean DataFrame
    dataf = pd.DataFrame(matchess, columns=["Date", "Time", "Day", "Venue"])
    return dataf

In [173]:
def clean_scrab(m_data,team):
    m_data.insert(0, "Team", team)
    df_homes = m_data[m_data["Venue"] == "Home"]
    df_homes['Date'] =  pd.to_datetime(df_homes['Date'])
    mask = df_homes["Date"].between("2022-10-31", "2024-10-31")
    df_homes = df_homes[mask]
    df_homes = df_homes.drop(columns= ["Venue"])
    return df_homes.reset_index(drop=True)

In [200]:
def enlarge_data_with(new_df):
    file_name = "big3_22kasım_24ekim.csv"
    try:
        old_df = pd.read_csv(file_name, index_col=0)
        old_df["Date"] = pd.to_datetime(old_df["Date"])
        old_df.drop(columns=[old_df.columns[0]])
        enlarged_df = pd.concat([new_df, old_df], axis=0)
    except FileNotFoundError:
        enlarged_df = new_df

    enlarged_df.reset_index(drop=True).to_csv(file_name)

In [197]:
def add_directly_to_data(plaintext, team):
    scrab = webscrab(plaintext)
    df = clean_scrab(scrab, team)
    enlarge_data_with(df)

In [198]:
def combine_dfs(df1, df2):
    # 1. Strip out any 'Unnamed' columns from both (if they exist)
    df1 = df1.loc[:, ~df1.columns.str.contains("Unnamed")]
    df2 = df2.loc[:, ~df2.columns.str.contains("Unnamed")]

    # 2. Stack them vertically
    combined = pd.concat([df1, df2], axis=0, ignore_index=True)

    # 3. Clean the Date column (strips timestamps like 00:00:00)
    combined["Date"] = combined["Date"].astype(str).str.split(" ").str[0]
    combined["Date"] = pd.to_datetime(combined["Date"])

    # 4. Remove duplicate matches and sort chronologically
    combined = (
        combined.drop_duplicates(subset=["Team", "Date"])
        .sort_values("Date")
        .reset_index(drop=True)
    )

    # Optional: Convert date back to a clean string format
    combined["Date"] = combined["Date"].dt.strftime("%Y-%m-%d")

    return combined

In [210]:
s = webscrab("""Date	Time	Comp	Round	Day	Venue	Result	GF	GA	Opponent	Poss	Attendance	Captain	Formation	Opp Formation	Referee	Match Report	Notes
2024-08-09	21:00	Süper Lig	Matchweek 1	Fri	Home	W	2	1	Hatayspor	71		Fernando Muslera	4-2-3-1	4-1-4-1	Turgut Doman	Match Report
2024-08-16	21:00	Süper Lig	Matchweek 2	Fri	Away	W	2	1	Konyaspor	60		Fernando Muslera	4-2-3-1	3-4-1-2	Direnç Tonusluoğlu	Match Report
2024-08-21	21:00 (22:00)	Champions Lg	Play-off round	Wed	Away	L	2	3	ch Young Boys	52	31,500	Fernando Muslera	4-2-3-1	4-4-2	Daniel Siebert	Match Report	Leg 1 of 2
2024-08-27	22:00	Champions Lg	Play-off round	Tue	Home	L	0	1	ch Young Boys	62	46,827	Fernando Muslera	4-4-2	4-2-3-1	Espen Eskås	Match Report	Leg 2 of 2; Young Boys won
2024-08-31	21:45	Süper Lig	Matchweek 4	Sat	Away	W	5	1	Adana Demirspor	58		Fernando Muslera	4-2-3-1	4-2-3-1	Kadir Sağlam	Match Report
2024-09-14	20:00	Süper Lig	Matchweek 5	Sat	Home	W	5	0	Rizespor	56		Fernando Muslera	4-2-3-1	5-4-1	Arda Kardeşler	Match Report
2024-09-17	20:00	Süper Lig	Matchweek 3	Tue	Home	W	3	1	Gaziantep FK	63		Fernando Muslera	4-2-3-1	5-4-1	Cihan Aydın	Match Report
2024-09-21	20:00	Süper Lig	Matchweek 6	Sat	Away	W	3	1	Fenerbahçe	44		Fernando Muslera	4-2-3-1	4-2-3-1	Atilla Karaoğlan	Match Report
2024-09-25	22:00	Europa Lg	League phase	Wed	Home	W	3	1	gr PAOK	56	50,700	Kaan Ayhan	4-2-3-1	4-2-3-1	Alejandro Hernández	Match Report
2024-09-28	19:00	Süper Lig	Matchweek 7	Sat	Home	D	3	3	Kasımpaşa	50		Fernando Muslera	4-4-2	4-1-4-1	Oğuzhan Çakir	Match Report
2024-10-03	19:45	Europa Lg	League phase	Thu	Away	D	2	2	lv FK Rīgas Futbola skola	76	8,502	Mauro Icardi	4-2-3-1	4-2-3-1	Filip Glova	Match Report
2024-10-06	19:00	Süper Lig	Matchweek 8	Sun	Home	W	1	0	Alanyaspor	55		Fernando Muslera	4-2-3-1	4-2-3-1	Mehmet Türkmen	Match Report
2024-10-19	19:00	Süper Lig	Matchweek 9	Sat	Away	W	3	0	Antalyaspor	51		Fernando Muslera	4-2-3-1	4-2-3-1	Ali Şansalan	Match Report
2024-10-23	17:30	Europa Lg	League phase	Wed	Home	W	4	3	se Elfsborg	66	36,938	Mauro Icardi	3-4-1-2	3-4-3	Matej Jug	Match Report
2024-10-28	20:00	Süper Lig	Matchweek 10	Mon	Home	W	2	1	Beşiktaş	39		Fernando Muslera	4-2-2-2	4-2-3-1	Arda Kardeşler	Match Report
2024-11-07	20:45	Europa Lg	League phase	Thu	Home	W	3	2	eng Tottenham Hotspur	54	51,739	Fernando Muslera	3-4-1-2	4-3-3	Lawrence Visser	Match Report
2024-11-10	16:00	Süper Lig	Matchweek 12	Sun	Home	W	3	2	Samsunspor	59		Fernando Muslera	4-2-3-1	4-2-3-1	Halil Umut Meler	Match Report
2024-11-23	19:00	Süper Lig	Matchweek 13	Sat	Away	W	1	0	BB Bodrumspor	67		Fernando Muslera	3-4-1-2	4-1-4-1	Muhammetali Metoğlu	Match Report
2024-11-28	18:45 (20:45)	Europa Lg	League phase	Thu	Away	D	1	1	nl AZ Alkmaar	60	16,229	Fernando Muslera	3-4-1-2	4-3-3	Horațiu Fesnic	Match Report
2024-12-01	19:00	Süper Lig	Matchweek 14	Sun	Home	D	2	2	Eyüpspor	66		Fernando Muslera	3-4-1-2	4-1-4-1	Oğuzhan Çakir	Match Report
2024-12-08	16:00	Süper Lig	Matchweek 15	Sun	Away	W	3	2	Sivasspor	51		Fernando Muslera	4-2-3-1	5-4-1	Turgut Doman	Match Report
2024-12-12	18:45 (20:45)	Europa Lg	League phase	Thu	Away	D	2	2	se Malmö	52	19,489	Fernando Muslera	3-4-3	4-4-2	John Brooks	Match Report
2024-12-16	21:00	Süper Lig	Matchweek 16	Mon	Home	W	4	3	Trabzonspor	58		Fernando Muslera	4-2-3-1	4-2-3-1	Kadir Sağlam	Match Report
2024-12-22	16:00	Süper Lig	Matchweek 17	Sun	Away	W	5	1	Kayserispor	50		Fernando Muslera	4-2-3-1	4-2-3-1	Mehmet Türkmen	Match Report
2025-01-04	19:00	Süper Lig	Matchweek 18	Sat	Home	W	2	1	Göztepe	59		Fernando Muslera	4-2-3-1	3-4-1-2	Alper Akarsu	Match Report
Date	Time	Comp	Round	Day	Venue	Result	GF	GA	Opponent	Poss	Attendance	Captain	Formation	Opp Formation	Referee	Match Report	Notes
2025-01-12	19:00	Süper Lig	Matchweek 19	Sun	Away	W	2	1	Başakşehir	48		Fernando Muslera	4-2-3-1	3-1-4-2	Atilla Karaoğlan	Match Report
2025-01-17	20:00	Süper Lig	Matchweek 20	Fri	Away	D	1	1	Hatayspor	61		Fernando Muslera	4-2-3-1	4-3-3	Cihan Aydın	Match Report
2025-01-21	18:30	Europa Lg	League phase	Tue	Home	D	3	3	ua Dynamo Kyiv	48	45,273	Fernando Muslera	4-2-3-1	4-3-3	Kristo Tohver	Match Report
2025-01-25	19:00	Süper Lig	Matchweek 21	Sat	Home	W	1	0	Konyaspor	58		Fernando Muslera	4-2-3-1	4-1-4-1	Halil Umut Meler	Match Report
2025-01-30	21:00 (23:00)	Europa Lg	League phase	Thu	Away	L	1	2	nl Ajax	61	54,100	Fernando Muslera	4-2-3-1	4-3-3	Jesús Gil	Match Report
2025-02-03	20:00	Süper Lig	Matchweek 22	Mon	Away	W	1	0	Gaziantep FK	52		Fernando Muslera	3-4-1-2	3-1-4-2	Kadir Sağlam	Match Report
2025-02-09	19:00	Süper Lig	Matchweek 23	Sun	Home	W	3	0	Adana Demirspor	66		Kaan Ayhan	3-4-1-2	4-2-3-1	Oğuzhan Çakir	Match Report	Match awarded to Galatasaray
2025-02-13	21:00 (23:00)	Europa Lg	Knockout phase play-offs	Thu	Away	L	1	4	nl AZ Alkmaar	45	14,092	Fernando Muslera	3-4-1-2	4-2-3-1	João Pinheiro	Match Report	Leg 1 of 2
2025-02-17	20:00	Süper Lig	Matchweek 24	Mon	Away	W	2	1	Rizespor	54		Fernando Muslera	3-4-1-2	4-2-3-1	Zorbay Küçük	Match Report
2025-02-20	20:45	Europa Lg	Knockout phase play-offs	Thu	Home	D	2	2	nl AZ Alkmaar	65	37,343	Abdülkerim Bardakcı	4-4-2	4-2-3-1	Anthony Taylor	Match Report	Leg 2 of 2; AZ Alkmaar won
2025-02-24	20:00	Süper Lig	Matchweek 25	Mon	Home	D	0	0	Fenerbahçe	60		Fernando Muslera	4-2-3-1	3-4-1-2	Slavko Vinčič	Match Report
2025-03-02	16:00	Süper Lig	Matchweek 26	Sun	Away	D	3	3	Kasımpaşa	53		Fernando Muslera	4-2-3-1	4-1-4-1	Cihan Aydın	Match Report
2025-03-09	20:30	Süper Lig	Matchweek 27	Sun	Away	W	2	1	Alanyaspor	65		Fernando Muslera	4-2-3-1	5-3-2	Yasin Kol	Match Report
2025-03-14	20:30	Süper Lig	Matchweek 28	Fri	Home	W	4	0	Antalyaspor	61		Fernando Muslera	4-2-3-1	5-4-1	Ozan Ergün	Match Report
2025-03-29	20:30	Süper Lig	Matchweek 29	Sat	Away	L	1	2	Beşiktaş	47		Fernando Muslera	4-2-3-1	4-2-3-1	Yasin Kol	Match Report
2025-04-11	20:00	Süper Lig	Matchweek 31	Fri	Away	W	2	0	Samsunspor	64		Abdülkerim Bardakcı	4-2-3-1	4-1-4-1	Ozan Ergün	Match Report
2025-04-18	20:00	Süper Lig	Matchweek 32	Fri	Home	W	2	0	BB Bodrumspor	71		Fernando Muslera	4-2-3-1	5-3-2	Mehmet Türkmen	Match Report
2025-04-27	19:00	Süper Lig	Matchweek 33	Sun	Away	W	5	1	Eyüpspor	50		Fernando Muslera	4-2-3-1	4-1-4-1	Çağdaş Altay	Match Report
2025-05-03	19:00	Süper Lig	Matchweek 34	Sat	Home	W	4	1	Sivasspor	72		Fernando Muslera	4-4-1-1	4-5-1	Ozan Ergün	Match Report
2025-05-10	19:00	Süper Lig	Matchweek 35	Sat	Away	W	2	0	Trabzonspor	58		Fernando Muslera	4-2-3-1	4-2-3-1	Atilla Karaoğlan	Match Report
2025-05-18	19:00	Süper Lig	Matchweek 36	Sun	Home	W	3	0	Kayserispor	59		Fernando Muslera	4-2-3-1	4-2-3-1	Ali Şansalan	Match Report
2025-05-24	19:00	Süper Lig	Matchweek 37	Sat	Away	W	2	0	Göztepe	64		Dries Mertens	4-3-3	3-4-1-2	Batuhan Kolak	Match Report
2025-05-30	20:00	Süper Lig	Matchweek 38	Fri	Home	W	2	0	Başakşehir	59		Fernando Muslera	4-2-3-1	3-1-4-2	Abdullah Taşkınsoy	Match Report	""")

In [211]:
c = clean_scrab(s, "Galatasaray")

In [212]:
c.to_csv('big3_22kasım_24ekim.csv')

In [213]:
add_directly_to_data("""Date	Time	Comp	Round	Day	Venue	Result	GF	GA	Opponent	Poss	Attendance	Captain	Formation	Opp Formation	Referee	Match Report	Notes
2023-07-25	19:00	Champions Lg	Second qualifying round	Tue	Away	D	2	2	lt Žalgiris		4,864	Fernando Muslera	4-2-3-1	4-4-2	Luís Godinho	Match Report	Leg 1 of 2
2023-08-02	21:30	Champions Lg	Second qualifying round	Wed	Home	W	1	0	lt Žalgiris		41,505	Fernando Muslera	4-2-3-1	4-4-2	Willy Delajod	Match Report	Leg 2 of 2; Galatasaray won
2023-08-08	21:00 (22:00)	Champions Lg	Third qualifying round	Tue	Away	W	3	0	si NK Olimpija Ljubljana		13,712	Fernando Muslera	4-2-3-1	5-4-1	Andreas Ekberg	Match Report	Leg 1 of 2
2023-08-12	21:45	Süper Lig	Matchweek 1	Sat	Away	D	0	0	Kayserispor	62		Fernando Muslera	4-2-3-1	4-4-1-1	Halil Umut Meler	Match Report
2023-08-15	21:00	Champions Lg	Third qualifying round	Tue	Home	W	1	0	si NK Olimpija Ljubljana		38,383	Fernando Muslera	4-2-3-1	5-4-1	Marco Guida	Match Report	Leg 2 of 2; Galatasaray won
2023-08-19	21:45	Süper Lig	Matchweek 2	Sat	Home	W	2	0	Trabzonspor	52		Fernando Muslera	4-2-3-1	4-1-4-1	Atilla Karaoğlan	Match Report
2023-08-23	21:00 (22:00)	Champions Lg	Play-off round	Wed	Away	W	3	2	no Molde	51	9,553	Fernando Muslera	4-4-1-1	3-5-2	Anthony Taylor	Match Report	Leg 1 of 2
2023-08-29	22:00	Champions Lg	Play-off round	Tue	Home	W	2	1	no Molde	60	47,845	Fernando Muslera	4-2-3-1	3-5-2	Szymon Marciniak	Match Report	Leg 2 of 2; Galatasaray won
2023-09-02	21:45	Süper Lig	Matchweek 4	Sat	Away	W	3	0	Gaziantep FK	64		Fernando Muslera	4-2-3-1	4-1-4-1	Ali Şansalan	Match Report
2023-09-16	20:00	Süper Lig	Matchweek 5	Sat	Home	W	4	2	Samsunspor	59		Fernando Muslera	4-2-3-1	4-4-1-1	Cihan Aydın	Match Report
2023-09-20	19:45	Champions Lg	Group stage	Wed	Home	D	2	2	dk FC Copenhagen	64	46,911	Fernando Muslera	4-2-3-1	4-3-3	Georgi Kabakov	Match Report
2023-09-23	20:00	Süper Lig	Matchweek 6	Sat	Away	W	2	1	Başakşehir	59		Fernando Muslera	4-2-3-1	4-1-4-1	Kadir Sağlam	Match Report
2023-09-26	20:00	Süper Lig	Matchweek 3	Tue	Away	W	1	0	İstanbulspor	70		Fernando Muslera	4-2-3-1	4-4-2	Abdulkadir Bitigen	Match Report
2023-09-30	19:00	Süper Lig	Matchweek 7	Sat	Home	W	2	1	Ankaragücü	68	41,513	Fernando Muslera	4-2-3-1	4-2-3-1	Volkan Bayarslan	Match Report
2023-10-03	20:00 (22:00)	Champions Lg	Group stage	Tue	Away	W	3	2	eng Manchester Utd	45	73,204	Fernando Muslera	4-2-3-1	4-3-3	Ivan Kružliak	Match Report
2023-10-07	19:00	Süper Lig	Matchweek 8	Sat	Away	W	2	0	Antalyaspor	58		Fernando Muslera	4-2-3-1	4-2-3-1	Zorbay Küçük	Match Report
2023-10-21	19:00	Süper Lig	Matchweek 9	Sat	Home	W	2	1	Beşiktaş	52	50,429	Fernando Muslera	4-2-3-1	4-3-3	Halil Umut Meler	Match Report
2023-10-24	19:45	Champions Lg	Group stage	Tue	Home	L	1	3	de Bayern Munich	52	51,776	Fernando Muslera	4-2-3-1	4-2-3-1	Davide Massa	Match Report
2023-10-28	19:00	Süper Lig	Matchweek 10	Sat	Away	W	1	0	Rizespor	62		Fernando Muslera	4-2-3-1	4-2-3-1	Kadir Sağlam	Match Report
2023-11-03	20:00	Süper Lig	Matchweek 11	Fri	Home	W	2	1	Kasımpaşa	63		Fernando Muslera	4-1-3-2	4-1-4-1	Arda Kardeşler	Match Report
2023-11-08	21:00 (23:00)	Champions Lg	Group stage	Wed	Away	L	1	2	de Bayern Munich	38	75,000	Fernando Muslera	4-2-3-1	4-2-3-1	António Nobre	Match Report
2023-11-11	19:00	Süper Lig	Matchweek 12	Sat	Away	L	1	2	Hatayspor	71		Fernando Muslera	4-2-3-1	4-1-4-1	Bahattin Şimşek	Match Report
2023-11-25	19:00	Süper Lig	Matchweek 13	Sat	Home	W	4	0	Alanyaspor	63		Fernando Muslera	4-2-3-1	4-2-3-1	Ali Şansalan	Match Report
2023-11-29	20:45	Champions Lg	Group stage	Wed	Home	D	3	3	eng Manchester Utd	58	51,733	Fernando Muslera	4-2-3-1	4-2-3-1	José Sánchez	Match Report
2023-12-02	19:00	Süper Lig	Matchweek 14	Sat	Away	W	2	0	Pendikspor	71		Fernando Muslera	4-2-3-1	4-2-3-1	Volkan Bayarslan	Match Report
Date	Time	Comp	Round	Day	Venue	Result	GF	GA	Opponent	Poss	Attendance	Captain	Formation	Opp Formation	Referee	Match Report	Notes
2023-12-08	20:00	Süper Lig	Matchweek 15	Fri	Home	W	3	1	Adana Demirspor	61		Fernando Muslera	4-2-3-1	4-2-3-1	Abdulkadir Bitigen	Match Report
2023-12-12	21:00 (23:00)	Champions Lg	Group stage	Tue	Away	L	0	1	dk FC Copenhagen	63	34,726	Fernando Muslera	4-2-3-1	4-3-3	Daniele Orsato	Match Report
2023-12-20	20:00	Süper Lig	Matchweek 17	Wed	Home	W	1	0	Fatih Karagümrük	54		Kerem Aktürkoğlu	4-2-3-1	4-2-3-1	Tugay Kaan Numanoğlu	Match Report
2023-12-24	19:00	Süper Lig	Matchweek 18	Sun	Away	D	0	0	Fenerbahçe	54		Fernando Muslera	4-2-3-1	4-2-3-1	Arda Kardeşler	Match Report
2024-01-07	19:00	Süper Lig	Matchweek 19	Sun	Home	W	3	0	Konyaspor	66		Fernando Muslera	4-2-3-1	4-2-3-1	Ali Şansalan	Match Report
2024-01-11	17:00	Süper Lig	Matchweek 16	Thu	Away	D	1	1	Sivasspor	63		Fernando Muslera	4-2-3-1	5-3-2	Zorbay Küçük	Match Report
2024-01-15	20:00	Süper Lig	Matchweek 20	Mon	Home	W	2	1	Kayserispor	59		Fernando Muslera	4-2-3-1	4-2-3-1	Atilla Karaoğlan	Match Report
2024-01-21	19:00	Süper Lig	Matchweek 21	Sun	Away	W	5	1	Trabzonspor	52		Fernando Muslera	4-2-3-1	4-2-3-1	Abdulkadir Bitigen	Match Report
2024-01-25	20:00	Süper Lig	Matchweek 22	Thu	Home	W	3	1	İstanbulspor	67		Fernando Muslera	4-2-3-1	5-4-1	Bahattin Şimşek	Match Report
2024-01-29	20:00	Süper Lig	Matchweek 23	Mon	Home	W	2	1	Gaziantep FK	76		Fernando Muslera	4-2-3-1	3-5-2	Arda Kardeşler	Match Report
2024-02-02	20:00	Süper Lig	Matchweek 24	Fri	Away	W	2	0	Samsunspor	55		Fernando Muslera	4-2-3-1	4-2-3-1	Tugay Kaan Numanoğlu	Match Report
2024-02-10	19:00	Süper Lig	Matchweek 25	Sat	Home	W	2	0	Başakşehir	53		Fernando Muslera	4-2-3-1	4-1-4-1	Cihan Aydın	Match Report
2024-02-15	20:45	Europa Lg	Knockout round play-offs	Thu	Home	W	3	2	cz Sparta Prague	54	46,583	Fernando Muslera	4-2-3-1	3-4-3	Alejandro Hernández	Match Report	Leg 1 of 2
2024-02-18	19:00	Süper Lig	Matchweek 26	Sun	Away	W	3	0	Ankaragücü	61		Fernando Muslera	4-2-3-1	4-3-3	Ali Şansalan	Match Report
2024-02-22	21:00 (23:00)	Europa Lg	Knockout round play-offs	Thu	Away	L	1	4	cz Sparta Prague	58	18,349	Fernando Muslera	4-2-3-1	3-4-1-2	Anthony Taylor	Match Report	Leg 2 of 2; Sparta Prague won
2024-02-26	20:00	Süper Lig	Matchweek 27	Mon	Home	W	2	1	Antalyaspor	53		Fernando Muslera	4-2-3-1	4-1-4-1	Abdulkadir Bitigen	Match Report
2024-03-03	19:00	Süper Lig	Matchweek 28	Sun	Away	W	1	0	Beşiktaş	52		Fernando Muslera	4-2-3-1	4-2-3-1	Halil Umut Meler	Match Report
2024-03-08	20:00	Süper Lig	Matchweek 29	Fri	Home	W	6	2	Rizespor	60		Fernando Muslera	4-2-3-1	4-2-3-1	Bahattin Şimşek	Match Report
2024-03-17	16:00	Süper Lig	Matchweek 30	Sun	Away	W	4	3	Kasımpaşa	60		Fernando Muslera	4-2-3-1	4-1-4-1	Atilla Karaoğlan	Match Report
2024-04-02	20:30	Süper Lig	Matchweek 31	Tue	Home	W	1	0	Hatayspor	58		Fernando Muslera	4-2-3-1	4-1-4-1	Ali Şansalan	Match Report
2024-04-15	20:00	Süper Lig	Matchweek 32	Mon	Away	W	4	0	Alanyaspor	55		Fernando Muslera	4-2-3-1	4-2-3-1	Cihan Aydın	Match Report
2024-04-21	19:00	Süper Lig	Matchweek 33	Sun	Home	W	4	1	Pendikspor	55		Fernando Muslera	4-2-3-1	4-2-3-1	Direnç Tonusluoğlu	Match Report
2024-04-26	20:00	Süper Lig	Matchweek 34	Fri	Away	W	3	0	Adana Demirspor	60		Fernando Muslera	4-2-3-1	4-2-3-1	Bahattin Şimşek	Match Report
2024-05-05	19:00	Süper Lig	Matchweek 35	Sun	Home	W	6	1	Sivasspor	69		Fernando Muslera	4-2-3-1	5-4-1	Arda Kardeşler	Match Report
2024-05-12	19:00	Süper Lig	Matchweek 36	Sun	Away	W	3	2	Fatih Karagümrük	65		Fernando Muslera	4-2-3-1	4-2-3-1	Volkan Bayarslan	Match Report
2024-05-19	19:00	Süper Lig	Matchweek 37	Sun	Home	L	0	1	Fenerbahçe	69		Fernando Muslera	4-2-3-1	4-2-3-1	Arda Kardeşler	Match Report
2024-05-26	19:00	Süper Lig	Matchweek 38	Sun	Away	W	3	1	Konyaspor	66		Fernando Muslera	4-2-3-1	4-2-3-1	Tugay Kaan Numanoğlu	Match Report""","Galatasaray")

In [214]:
add_directly_to_data("""Date	Time	Comp	Round	Day	Venue	Result	GF	GA	Opponent	Poss	Attendance	Captain	Formation	Opp Formation	Referee	Match Report	Notes
2023-07-25	19:00	Champions Lg	Second qualifying round	Tue	Away	D	2	2	lt Žalgiris		4,864	Fernando Muslera	4-2-3-1	4-4-2	Luís Godinho	Match Report	Leg 1 of 2
2023-08-02	21:30	Champions Lg	Second qualifying round	Wed	Home	W	1	0	lt Žalgiris		41,505	Fernando Muslera	4-2-3-1	4-4-2	Willy Delajod	Match Report	Leg 2 of 2; Galatasaray won
2023-08-08	21:00 (22:00)	Champions Lg	Third qualifying round	Tue	Away	W	3	0	si NK Olimpija Ljubljana		13,712	Fernando Muslera	4-2-3-1	5-4-1	Andreas Ekberg	Match Report	Leg 1 of 2
2023-08-12	21:45	Süper Lig	Matchweek 1	Sat	Away	D	0	0	Kayserispor	62		Fernando Muslera	4-2-3-1	4-4-1-1	Halil Umut Meler	Match Report
2023-08-15	21:00	Champions Lg	Third qualifying round	Tue	Home	W	1	0	si NK Olimpija Ljubljana		38,383	Fernando Muslera	4-2-3-1	5-4-1	Marco Guida	Match Report	Leg 2 of 2; Galatasaray won
2023-08-19	21:45	Süper Lig	Matchweek 2	Sat	Home	W	2	0	Trabzonspor	52		Fernando Muslera	4-2-3-1	4-1-4-1	Atilla Karaoğlan	Match Report
2023-08-23	21:00 (22:00)	Champions Lg	Play-off round	Wed	Away	W	3	2	no Molde	51	9,553	Fernando Muslera	4-4-1-1	3-5-2	Anthony Taylor	Match Report	Leg 1 of 2
2023-08-29	22:00	Champions Lg	Play-off round	Tue	Home	W	2	1	no Molde	60	47,845	Fernando Muslera	4-2-3-1	3-5-2	Szymon Marciniak	Match Report	Leg 2 of 2; Galatasaray won
2023-09-02	21:45	Süper Lig	Matchweek 4	Sat	Away	W	3	0	Gaziantep FK	64		Fernando Muslera	4-2-3-1	4-1-4-1	Ali Şansalan	Match Report
2023-09-16	20:00	Süper Lig	Matchweek 5	Sat	Home	W	4	2	Samsunspor	59		Fernando Muslera	4-2-3-1	4-4-1-1	Cihan Aydın	Match Report
2023-09-20	19:45	Champions Lg	Group stage	Wed	Home	D	2	2	dk FC Copenhagen	64	46,911	Fernando Muslera	4-2-3-1	4-3-3	Georgi Kabakov	Match Report
2023-09-23	20:00	Süper Lig	Matchweek 6	Sat	Away	W	2	1	Başakşehir	59		Fernando Muslera	4-2-3-1	4-1-4-1	Kadir Sağlam	Match Report
2023-09-26	20:00	Süper Lig	Matchweek 3	Tue	Away	W	1	0	İstanbulspor	70		Fernando Muslera	4-2-3-1	4-4-2	Abdulkadir Bitigen	Match Report
2023-09-30	19:00	Süper Lig	Matchweek 7	Sat	Home	W	2	1	Ankaragücü	68	41,513	Fernando Muslera	4-2-3-1	4-2-3-1	Volkan Bayarslan	Match Report
2023-10-03	20:00 (22:00)	Champions Lg	Group stage	Tue	Away	W	3	2	eng Manchester Utd	45	73,204	Fernando Muslera	4-2-3-1	4-3-3	Ivan Kružliak	Match Report
2023-10-07	19:00	Süper Lig	Matchweek 8	Sat	Away	W	2	0	Antalyaspor	58		Fernando Muslera	4-2-3-1	4-2-3-1	Zorbay Küçük	Match Report
2023-10-21	19:00	Süper Lig	Matchweek 9	Sat	Home	W	2	1	Beşiktaş	52	50,429	Fernando Muslera	4-2-3-1	4-3-3	Halil Umut Meler	Match Report
2023-10-24	19:45	Champions Lg	Group stage	Tue	Home	L	1	3	de Bayern Munich	52	51,776	Fernando Muslera	4-2-3-1	4-2-3-1	Davide Massa	Match Report
2023-10-28	19:00	Süper Lig	Matchweek 10	Sat	Away	W	1	0	Rizespor	62		Fernando Muslera	4-2-3-1	4-2-3-1	Kadir Sağlam	Match Report
2023-11-03	20:00	Süper Lig	Matchweek 11	Fri	Home	W	2	1	Kasımpaşa	63		Fernando Muslera	4-1-3-2	4-1-4-1	Arda Kardeşler	Match Report
2023-11-08	21:00 (23:00)	Champions Lg	Group stage	Wed	Away	L	1	2	de Bayern Munich	38	75,000	Fernando Muslera	4-2-3-1	4-2-3-1	António Nobre	Match Report
2023-11-11	19:00	Süper Lig	Matchweek 12	Sat	Away	L	1	2	Hatayspor	71		Fernando Muslera	4-2-3-1	4-1-4-1	Bahattin Şimşek	Match Report
2023-11-25	19:00	Süper Lig	Matchweek 13	Sat	Home	W	4	0	Alanyaspor	63		Fernando Muslera	4-2-3-1	4-2-3-1	Ali Şansalan	Match Report
2023-11-29	20:45	Champions Lg	Group stage	Wed	Home	D	3	3	eng Manchester Utd	58	51,733	Fernando Muslera	4-2-3-1	4-2-3-1	José Sánchez	Match Report
2023-12-02	19:00	Süper Lig	Matchweek 14	Sat	Away	W	2	0	Pendikspor	71		Fernando Muslera	4-2-3-1	4-2-3-1	Volkan Bayarslan	Match Report
Date	Time	Comp	Round	Day	Venue	Result	GF	GA	Opponent	Poss	Attendance	Captain	Formation	Opp Formation	Referee	Match Report	Notes
2023-12-08	20:00	Süper Lig	Matchweek 15	Fri	Home	W	3	1	Adana Demirspor	61		Fernando Muslera	4-2-3-1	4-2-3-1	Abdulkadir Bitigen	Match Report
2023-12-12	21:00 (23:00)	Champions Lg	Group stage	Tue	Away	L	0	1	dk FC Copenhagen	63	34,726	Fernando Muslera	4-2-3-1	4-3-3	Daniele Orsato	Match Report
2023-12-20	20:00	Süper Lig	Matchweek 17	Wed	Home	W	1	0	Fatih Karagümrük	54		Kerem Aktürkoğlu	4-2-3-1	4-2-3-1	Tugay Kaan Numanoğlu	Match Report
2023-12-24	19:00	Süper Lig	Matchweek 18	Sun	Away	D	0	0	Fenerbahçe	54		Fernando Muslera	4-2-3-1	4-2-3-1	Arda Kardeşler	Match Report
2024-01-07	19:00	Süper Lig	Matchweek 19	Sun	Home	W	3	0	Konyaspor	66		Fernando Muslera	4-2-3-1	4-2-3-1	Ali Şansalan	Match Report
2024-01-11	17:00	Süper Lig	Matchweek 16	Thu	Away	D	1	1	Sivasspor	63		Fernando Muslera	4-2-3-1	5-3-2	Zorbay Küçük	Match Report
2024-01-15	20:00	Süper Lig	Matchweek 20	Mon	Home	W	2	1	Kayserispor	59		Fernando Muslera	4-2-3-1	4-2-3-1	Atilla Karaoğlan	Match Report
2024-01-21	19:00	Süper Lig	Matchweek 21	Sun	Away	W	5	1	Trabzonspor	52		Fernando Muslera	4-2-3-1	4-2-3-1	Abdulkadir Bitigen	Match Report
2024-01-25	20:00	Süper Lig	Matchweek 22	Thu	Home	W	3	1	İstanbulspor	67		Fernando Muslera	4-2-3-1	5-4-1	Bahattin Şimşek	Match Report
2024-01-29	20:00	Süper Lig	Matchweek 23	Mon	Home	W	2	1	Gaziantep FK	76		Fernando Muslera	4-2-3-1	3-5-2	Arda Kardeşler	Match Report
2024-02-02	20:00	Süper Lig	Matchweek 24	Fri	Away	W	2	0	Samsunspor	55		Fernando Muslera	4-2-3-1	4-2-3-1	Tugay Kaan Numanoğlu	Match Report
2024-02-10	19:00	Süper Lig	Matchweek 25	Sat	Home	W	2	0	Başakşehir	53		Fernando Muslera	4-2-3-1	4-1-4-1	Cihan Aydın	Match Report
2024-02-15	20:45	Europa Lg	Knockout round play-offs	Thu	Home	W	3	2	cz Sparta Prague	54	46,583	Fernando Muslera	4-2-3-1	3-4-3	Alejandro Hernández	Match Report	Leg 1 of 2
2024-02-18	19:00	Süper Lig	Matchweek 26	Sun	Away	W	3	0	Ankaragücü	61		Fernando Muslera	4-2-3-1	4-3-3	Ali Şansalan	Match Report
2024-02-22	21:00 (23:00)	Europa Lg	Knockout round play-offs	Thu	Away	L	1	4	cz Sparta Prague	58	18,349	Fernando Muslera	4-2-3-1	3-4-1-2	Anthony Taylor	Match Report	Leg 2 of 2; Sparta Prague won
2024-02-26	20:00	Süper Lig	Matchweek 27	Mon	Home	W	2	1	Antalyaspor	53		Fernando Muslera	4-2-3-1	4-1-4-1	Abdulkadir Bitigen	Match Report
2024-03-03	19:00	Süper Lig	Matchweek 28	Sun	Away	W	1	0	Beşiktaş	52		Fernando Muslera	4-2-3-1	4-2-3-1	Halil Umut Meler	Match Report
2024-03-08	20:00	Süper Lig	Matchweek 29	Fri	Home	W	6	2	Rizespor	60		Fernando Muslera	4-2-3-1	4-2-3-1	Bahattin Şimşek	Match Report
2024-03-17	16:00	Süper Lig	Matchweek 30	Sun	Away	W	4	3	Kasımpaşa	60		Fernando Muslera	4-2-3-1	4-1-4-1	Atilla Karaoğlan	Match Report
2024-04-02	20:30	Süper Lig	Matchweek 31	Tue	Home	W	1	0	Hatayspor	58		Fernando Muslera	4-2-3-1	4-1-4-1	Ali Şansalan	Match Report
2024-04-15	20:00	Süper Lig	Matchweek 32	Mon	Away	W	4	0	Alanyaspor	55		Fernando Muslera	4-2-3-1	4-2-3-1	Cihan Aydın	Match Report
2024-04-21	19:00	Süper Lig	Matchweek 33	Sun	Home	W	4	1	Pendikspor	55		Fernando Muslera	4-2-3-1	4-2-3-1	Direnç Tonusluoğlu	Match Report
2024-04-26	20:00	Süper Lig	Matchweek 34	Fri	Away	W	3	0	Adana Demirspor	60		Fernando Muslera	4-2-3-1	4-2-3-1	Bahattin Şimşek	Match Report
2024-05-05	19:00	Süper Lig	Matchweek 35	Sun	Home	W	6	1	Sivasspor	69		Fernando Muslera	4-2-3-1	5-4-1	Arda Kardeşler	Match Report
2024-05-12	19:00	Süper Lig	Matchweek 36	Sun	Away	W	3	2	Fatih Karagümrük	65		Fernando Muslera	4-2-3-1	4-2-3-1	Volkan Bayarslan	Match Report
2024-05-19	19:00	Süper Lig	Matchweek 37	Sun	Home	L	0	1	Fenerbahçe	69		Fernando Muslera	4-2-3-1	4-2-3-1	Arda Kardeşler	Match Report
2024-05-26	19:00	Süper Lig	Matchweek 38	Sun	Away	W	3	1	Konyaspor	66		Fernando Muslera	4-2-3-1	4-2-3-1	Tugay Kaan Numanoğlu	Match Report""","Galatasaray")

In [215]:
add_directly_to_data("""Date	Time	Round	Day	Venue	Result	GF	GA	Opponent	Poss	Attendance	Captain	Formation	Opp Formation	Referee	Match Report	Notes
2022-08-07	21:45	Matchweek 1	Sun	Away	W	1	0	Antalyaspor	56		Fernando Muslera	4-2-3-1	4-2-3-1	Atilla Karaoğlan	Match Report
2022-08-13	21:45	Matchweek 2	Sat	Home	L	0	1	Giresunspor	66	45,484	Fernando Muslera	4-2-3-1	4-2-3-1	Kadir Sağlam	Match Report
2022-08-19	21:00	Matchweek 3	Fri	Away	W	1	0	Ümraniyespor	60		Fernando Muslera	4-2-3-1	4-1-4-1	Ali Şansalan	Match Report
2022-08-28	21:45	Matchweek 4	Sun	Away	D	0	0	Trabzonspor	57		Fernando Muslera	4-2-3-1	4-2-3-1	Ali Palabıyık	Match Report
2022-09-05	21:00	Matchweek 5	Mon	Home	W	2	1	Gaziantep FK	58	44,061	Fernando Muslera	4-2-3-1	4-4-1-1	Yaşar Kemal Uğurlu	Match Report
2022-09-11	20:00	Matchweek 6	Sun	Away	W	3	2	Kasımpaşa	63		Fernando Muslera	4-2-3-1	4-1-4-1	Volkan Bayarslan	Match Report
2022-09-16	20:00	Matchweek 7	Fri	Home	W	2	1	Konyaspor	55		Fernando Muslera	4-2-3-1	4-1-4-1	Abdulkadir Bitigen	Match Report
2022-10-01	20:00	Matchweek 8	Sat	Away	D	0	0	Adana Demirspor	55		Fernando Muslera	4-2-3-1	4-2-3-1	Halil Umut Meler	Match Report
2022-10-15	20:00	Matchweek 10	Sat	Away	L	1	2	Kayserispor	61		Fernando Muslera	4-2-3-1	4-1-4-1	Arda Kardeşler	Match Report
2022-10-23	20:00	Matchweek 11	Sun	Home	D	2	2	Alanyaspor	34		Fernando Muslera	4-2-3-1	4-2-3-1	Ali Palabıyık	Match Report
2022-10-28	20:00	Matchweek 12	Fri	Away	W	2	0	Fatih Karagümrük	70		Fernando Muslera	4-2-3-1	4-3-3	Bahattin Şimşek	Match Report
2022-11-05	20:00	Matchweek 13	Sat	Home	W	2	1	Beşiktaş	54		Fernando Muslera	4-2-3-1	4-3-3	Halil Umut Meler	Match Report
2022-11-12	20:00	Matchweek 14	Sat	Away	W	7	0	Başakşehir	40		Fernando Muslera	4-2-3-1	4-1-4-1	Atilla Karaoğlan	Match Report
2022-12-25	19:00	Matchweek 15	Sun	Home	W	2	1	İstanbulspor	48		Bafétimbi Gomis	4-2-3-1	4-1-4-1	Abdulkadir Bitigen	Match Report
2022-12-29	17:00	Matchweek 16	Thu	Away	W	2	1	Sivasspor	53		Fernando Muslera	4-2-3-1	4-3-3	Erkan Özdamar	Match Report
2023-01-04	20:00	Matchweek 17	Wed	Home	W	2	1	Ankaragücü	44		Fernando Muslera	4-2-3-1	4-1-4-1	Zorbay Küçük	Match Report
2023-01-08	19:00	Matchweek 18	Sun	Away	W	3	0	Fenerbahçe	37		Fernando Muslera	4-2-3-1	3-4-3	Halil Umut Meler	Match Report
2023-01-13	20:00	Matchweek 19	Fri	Home	W	4	0	Hatayspor	66		Fernando Muslera	4-2-3-1	4-2-3-1	Kadir Sağlam	Match Report
2023-01-21	19:00	Matchweek 20	Sat	Home	W	2	1	Antalyaspor	60		Fernando Muslera	4-2-3-1	4-2-3-1	Ali Şansalan	Match Report
2023-01-28	19:00	Matchweek 21	Sat	Away	W	4	0	Giresunspor	57		Fernando Muslera	4-2-3-1	4-4-2	Bahattin Şimşek	Match Report
2023-02-01	20:00	Matchweek 22	Wed	Home	W	3	2	Ümraniyespor	67	43,823	Fernando Muslera	4-2-3-1	4-3-3	Cihan Aydın	Match Report
2023-02-05	19:00	Matchweek 23	Sun	Home	W	2	1	Trabzonspor	53		Fernando Muslera	4-2-3-1	4-1-4-1	Halil Umut Meler	Match Report
2023-03-05		Matchweek 24	Sun	Away	W	3	0	Gaziantep FK							Match Report	Match awarded to Galatasaray
2023-03-11	19:00	Matchweek 25	Sat	Home	W	1	0	Kasımpaşa	69		Fernando Muslera	4-2-3-1	4-4-2	Atilla Karaoğlan	Match Report
2023-03-17	20:00	Matchweek 26	Fri	Away	L	1	2	Konyaspor	64		Fernando Muslera	4-2-3-1	4-2-3-1	Zorbay Küçük	Match Report
Date	Time	Round	Day	Venue	Result	GF	GA	Opponent	Poss	Attendance	Captain	Formation	Opp Formation	Referee	Match Report	Notes
2023-04-01	20:30	Matchweek 27	Sat	Home	W	2	0	Adana Demirspor	59		Fernando Muslera	4-2-3-1	4-4-2	Ali Şansalan	Match Report
2023-04-14	20:30	Matchweek 29	Fri	Home	W	6	0	Kayserispor	66		Fernando Muslera	4-2-3-1	4-2-3-1	Mustafa Kürşad Filiz	Match Report
2023-04-18	20:30	Matchweek 30	Tue	Away	W	4	1	Alanyaspor	56		Fernando Muslera	4-2-3-1	4-3-3	Abdulkadir Bitigen	Match Report
2023-04-23	19:00	Matchweek 31	Sun	Home	D	3	3	Fatih Karagümrük	65	45,101	Fernando Muslera	4-2-3-1	4-3-3	Volkan Bayarslan	Match Report
2023-04-30	19:00	Matchweek 32	Sun	Away	L	1	3	Beşiktaş	55		Fernando Muslera	4-2-3-1	4-2-3-1	Ali Şansalan	Match Report
2023-05-08	20:00	Matchweek 33	Mon	Home	W	1	0	Başakşehir	60		Fernando Muslera	4-2-3-1	4-2-3-1	Halil Umut Meler	Match Report
2023-05-16	20:00	Matchweek 34	Tue	Away	W	2	0	İstanbulspor	57		Fernando Muslera	4-2-3-1	4-3-2-1	Tugay Kaan Numanoğlu	Match Report
2023-05-20	19:00	Matchweek 35	Sat	Home	W	2	0	Sivasspor	65		Fernando Muslera	4-2-3-1	4-1-4-1	Arda Kardeşler	Match Report
2023-05-30	20:00	Matchweek 36	Tue	Away	W	4	1	Ankaragücü	53		Fernando Muslera	4-2-3-1	4-3-3	Volkan Bayarslan	Match Report
2023-06-04	19:00	Matchweek 37	Sun	Home	W	3	0	Fenerbahçe	52		Fernando Muslera	4-2-3-1	4-1-3-2	Abdulkadir Bitigen	Match Report
2023-06-07		Matchweek 38	Wed	Away	W	3	0	Hatayspor							Match Report	Match awarded to Gala""","Galatasaray")

In [216]:
add_directly_to_data("""Date	Time	Comp	Round	Day	Venue	Result	GF	GA	Opponent	Poss	Attendance	Captain	Formation	Opp Formation	Referee	Match Report	Notes
2024-08-11	21:45	Süper Lig	Matchweek 1	Sun	Away	W	2	0	Samsunspor	42		Mert Günok	4-2-3-1	4-1-4-1	Ali Şansalan	Match Report
2024-08-18	21:45	Süper Lig	Matchweek 2	Sun	Home	W	4	2	Antalyaspor	56		Mert Günok	4-2-3-1	3-5-2	Mehmet Türkmen	Match Report
2024-08-22	20:30 (21:30)	Europa Lg	Play-off round	Thu	Away	D	3	3	ch Lugano		5,270	Mert Günok	4-2-3-1	4-3-3	Morten Krogh	Match Report	Leg 1 of 2
2024-08-29	21:00	Europa Lg	Play-off round	Thu	Home	W	5	1	ch Lugano		35,370	Mert Günok	4-2-3-1	4-3-3	Radu Petrescu	Match Report	Leg 2 of 2; Beşiktaş won
2024-09-01	21:45	Süper Lig	Matchweek 4	Sun	Home	W	2	0	Sivasspor	67		Mert Günok	4-2-3-1	5-4-1	Direnç Tonusluoğlu	Match Report
2024-09-15	20:00	Süper Lig	Matchweek 5	Sun	Away	D	1	1	Trabzonspor	76		Mert Günok	4-2-3-1	4-2-3-1	Ali Şansalan	Match Report
2024-09-22	20:00	Süper Lig	Matchweek 6	Sun	Home	W	2	1	Eyüpspor	62		Mert Günok	3-4-3	4-1-4-1	Zorbay Küçük	Match Report
2024-09-26	21:00 (22:00)	Europa Lg	League phase	Thu	Away	L	0	4	nl Ajax	53	54,010	Mert Günok	4-2-3-1	4-3-3	John Brooks	Match Report
2024-09-30	20:00	Süper Lig	Matchweek 7	Mon	Away	W	3	0	Kayserispor	52		Mert Günok	4-2-3-1	4-1-4-1	Atilla Karaoğlan	Match Report
2024-10-03	22:00	Europa Lg	League phase	Thu	Home	L	1	3	de Eintracht Frankfurt	65	35,305	Ersin Destanoğlu	4-2-3-1	4-4-2	John Beaton	Match Report
2024-10-06	19:00	Süper Lig	Matchweek 8	Sun	Away	D	1	1	Gaziantep FK	58		Mert Günok	4-2-3-1	4-1-4-1	Muhammetali Metoğlu	Match Report
2024-10-20	16:00	Süper Lig	Matchweek 9	Sun	Home	W	2	0	Konyaspor	56		Mert Günok	4-2-3-1	4-2-3-1	Kadir Sağlam	Match Report
2024-10-24	21:00 (22:00)	Europa Lg	League phase	Thu	Away	W	1	0	fr Lyon	42	27,350	Ersin Destanoğlu	4-2-3-1	4-2-3-1	Harm Osmers	Match Report
2024-10-28	20:00	Süper Lig	Matchweek 10	Mon	Away	L	1	2	Galatasaray	61		Ersin Destanoğlu	4-2-3-1	4-2-2-2	Arda Kardeşler	Match Report
2024-11-02	19:00	Süper Lig	Matchweek 11	Sat	Home	L	1	3	Kasımpaşa	55		Mert Günok	4-2-3-1	4-1-4-1	Zorbay Küçük	Match Report
2024-11-06	18:30	Europa Lg	League phase	Wed	Home	W	2	1	se Malmö	54	24,652	Mert Günok	4-2-3-1	4-2-3-1	Sven Jablonski	Match Report
2024-11-10	19:00	Süper Lig	Matchweek 12	Sun	Away	D	0	0	Başakşehir	61		Mert Günok	4-2-3-1	4-1-4-1	Turgut Doman	Match Report
2024-11-24	19:00	Süper Lig	Matchweek 13	Sun	Home	L	2	4	Göztepe	63		Mert Günok	3-4-3	5-3-2	Alper Akarsu	Match Report
2024-11-28	20:45	Europa Lg	League phase	Thu	Home	L	1	3	il Maccabi Tel Aviv FC	59	20,124	Mert Günok	4-2-3-1	4-2-3-1	Simone Sozza	Match Report
2024-12-02	20:00	Süper Lig	Matchweek 14	Mon	Away	D	1	1	Hatayspor	56		Mert Günok	4-2-3-1	4-2-3-1	Kadir Sağlam	Match Report
2024-12-07	19:00	Süper Lig	Matchweek 15	Sat	Home	W	1	0	Fenerbahçe	49		Mert Günok	4-2-3-1	4-2-3-1	Mehmet Türkmen	Match Report
2024-12-12	21:00 (23:00)	Europa Lg	League phase	Thu	Away	L	1	2	no Bodø/Glimt	41	5,813	Ersin Destanoğlu	4-2-3-1	4-3-3	Filip Glova	Match Report
2024-12-16	19:00	Süper Lig	Matchweek 16	Mon	Away	L	1	2	Adana Demirspor	65		Mert Günok	4-2-3-1	4-2-3-1	Oğuzhan Çakir	Match Report
2024-12-21	19:00	Süper Lig	Matchweek 17	Sat	Home	D	1	1	Alanyaspor	56		Mert Günok	4-2-3-1	4-1-4-1	Cihan Aydın	Match Report
2025-01-03	20:00	Süper Lig	Matchweek 18	Fri	Away	D	1	1	Rizespor	53		Mert Günok	4-2-3-1	4-2-3-1	Ozan Ergün	Match Report
Date	Time	Comp	Round	Day	Venue	Result	GF	GA	Opponent	Poss	Attendance	Captain	Formation	Opp Formation	Referee	Match Report	Notes
2025-01-11	19:00	Süper Lig	Matchweek 19	Sat	Home	W	2	1	BB Bodrumspor	52		Mert Günok	4-2-3-1	4-1-4-1	Mehmet Türkmen	Match Report
2025-01-18	19:00	Süper Lig	Matchweek 20	Sat	Home	D	0	0	Samsunspor	66		Mert Günok	4-2-3-1	4-2-3-1	Kadir Sağlam	Match Report
2025-01-22	18:30	Europa Lg	League phase	Wed	Home	W	4	1	es Athletic Club	47	35,980	Mert Günok	4-2-3-1	4-2-3-1	Julian Weinberger	Match Report
2025-01-26	16:00	Süper Lig	Matchweek 21	Sun	Away	D	1	1	Antalyaspor	54		Mert Günok	4-2-3-1	4-4-2	Oğuzhan Çakir	Match Report
2025-01-30	21:00 (23:00)	Europa Lg	League phase	Thu	Away	L	0	1	nl Twente	51	28,500	Mert Günok	4-2-3-1	4-2-3-1	Szymon Marciniak	Match Report
2025-02-08	19:00	Süper Lig	Matchweek 23	Sat	Away	W	2	0	Sivasspor	49		Mert Günok	4-2-3-1	4-3-3	Ali Şansalan	Match Report
2025-02-15	19:00	Süper Lig	Matchweek 24	Sat	Home	W	2	1	Trabzonspor	49		Mert Günok	4-2-3-1	4-2-3-1	Cihan Aydın	Match Report
2025-02-21	20:00	Süper Lig	Matchweek 25	Fri	Away	W	3	1	Eyüpspor	55		Mert Günok	4-2-3-1	4-1-4-1	Muhammetali Metoğlu	Match Report
2025-03-01	20:30	Süper Lig	Matchweek 26	Sat	Home	W	2	0	Kayserispor	49		Mert Günok	4-2-3-1	4-2-3-1	Halil Umut Meler	Match Report
2025-03-10	20:30	Süper Lig	Matchweek 27	Mon	Home	L	1	2	Gaziantep FK	59		Mert Günok	4-2-3-1	3-5-2	Zorbay Küçük	Match Report
2025-03-15	20:30	Süper Lig	Matchweek 28	Sat	Away	L	0	1	Konyaspor	66		Mert Günok	4-2-3-1	3-4-3	Ali Şansalan	Match Report
2025-03-29	20:30	Süper Lig	Matchweek 29	Sat	Home	W	2	1	Galatasaray	53		Mert Günok	4-2-3-1	4-2-3-1	Yasin Kol	Match Report
2025-04-07	20:00	Süper Lig	Matchweek 30	Mon	Away	D	1	1	Kasımpaşa	47		Mert Günok	4-2-3-1	3-5-2	Kadir Sağlam	Match Report
2025-04-12	19:00	Süper Lig	Matchweek 31	Sat	Home	L	0	2	Başakşehir	56		Mert Günok	4-2-3-1	4-1-4-1	Cihan Aydın	Match Report
2025-04-19	19:00	Süper Lig	Matchweek 32	Sat	Away	D	1	1	Göztepe	56		Mert Günok	4-1-4-1	3-4-1-2	Ozan Ergün	Match Report
2025-04-25	20:00	Süper Lig	Matchweek 33	Fri	Home	W	5	1	Hatayspor	52		Mert Günok	4-2-3-1	4-2-3-1	Adnan Deniz Kayatepe	Match Report
2025-05-04	19:00	Süper Lig	Matchweek 34	Sun	Away	W	1	0	Fenerbahçe	38		Mert Günok	4-2-3-1	4-4-2	Yasin Kol	Match Report
2025-05-11	19:00	Süper Lig	Matchweek 35	Sun	Home	W	4	1	Adana Demirspor	63		Ersin Destanoğlu	4-2-3-1	5-3-2	Çağdaş Altay	Match Report
2025-05-18	19:00	Süper Lig	Matchweek 36	Sun	Away	D	1	1	Alanyaspor	60		Mert Günok	4-2-3-1	3-4-3	Mehmet Türkmen	Match Report
2025-05-25	19:00	Süper Lig	Matchweek 37	Sun	Home	L	1	2	Rizespor	58		Mert Günok	4-2-3-1	4-2-3-1	Halil Umut Meler	Match Report
2025-06-01	19:00	Süper Lig	Matchweek 38	Sun	Away	W	4	0	BB Bodrumspor	60		Mert Günok	4-3-3	4-2-3-1	Cihan Aydın	Match Report	""","Beşiktaş")

In [217]:
add_directly_to_data("""Date	Time	Comp	Round	Day	Venue	Result	GF	GA	Opponent	Poss	Attendance	Captain	Formation	Opp Formation	Referee	Match Report	Notes
2023-07-27	21:00	Conf Lg	Second qualifying round	Thu	Home	W	3	1	al KF Tirana		22,820	Necip Uysal	4-1-4-1	4-2-3-1	Daniel Schlager	Match Report	Leg 1 of 2
2023-08-03	20:00 (21:00)	Conf Lg	Second qualifying round	Thu	Away	W	2	0	al KF Tirana		10,865	Necip Uysal	4-1-4-1	4-3-1-2	Stuart Attwell	Match Report	Leg 2 of 2; Beşiktaş won
2023-08-10	21:00 (20:00)	Conf Lg	Third qualifying round	Thu	Away	W	3	1	az Neftçi		11,000	Mert Günok	4-3-3	4-3-3	Lukas Fähndrich	Match Report	Leg 1 of 2
2023-08-14	21:00	Süper Lig	Matchweek 1	Mon	Away	W	1	0	Fatih Karagümrük	65		Necip Uysal	4-3-3	4-3-3	Erkan Özdamar	Match Report
2023-08-17	21:00	Conf Lg	Third qualifying round	Thu	Home	W	2	1	az Neftçi		21,890	Necip Uysal	4-1-4-1	4-3-3	Julian Weinberger	Match Report	Leg 2 of 2; Beşiktaş won
2023-08-20	21:45	Süper Lig	Matchweek 2	Sun	Home	D	1	1	Pendikspor	61		Mert Günok	4-2-3-1	4-5-1	Kadir Sağlam	Match Report
2023-08-24	20:00	Conf Lg	Play-off round	Thu	Away	W	3	2	ua Dynamo Kyiv			Mert Günok	4-1-4-1	4-4-1-1	Darren England	Match Report	Leg 1 of 2
2023-08-31	21:00	Conf Lg	Play-off round	Thu	Home	W	1	0	ua Dynamo Kyiv		33,376	Necip Uysal	4-1-4-1	4-4-1-1	João Pinheiro	Match Report	Leg 2 of 2; Beşiktaş won
2023-09-03	19:15	Süper Lig	Matchweek 4	Sun	Home	W	2	0	Sivasspor	66		Necip Uysal	4-2-3-1	4-3-3	Zorbay Küçük	Match Report
2023-09-17	20:00	Süper Lig	Matchweek 5	Sun	Away	L	0	3	Trabzonspor	71		Necip Uysal	4-2-3-1	4-2-3-1	Volkan Bayarslan	Match Report
2023-09-21	21:00 (22:00)	Conf Lg	Group stage	Thu	Away	D	1	1	be Club Brugge		18,442	Mert Günok	4-2-3-1	4-3-3	Aliaksei Kulbakou	Match Report
2023-09-24	17:00	Süper Lig	Matchweek 6	Sun	Home	W	2	1	Kayserispor	63		Cenk Tosun	4-1-4-1	4-1-4-1	Ali Şansalan	Match Report
2023-09-27	20:00	Süper Lig	Matchweek 3	Wed	Away	L	2	4	Adana Demirspor	62		Necip Uysal	4-1-4-1	4-3-3	Umit Öztürk	Match Report
2023-10-01	19:00	Süper Lig	Matchweek 7	Sun	Away	W	2	0	Konyaspor	61		Vincent Aboubakar	4-4-2	4-2-3-1	Arda Kardeşler	Match Report
2023-10-05	19:45	Conf Lg	Group stage	Thu	Home	L	2	3	ch Lugano		24,648	Cenk Tosun	4-1-4-1	4-3-3	David Fuxman	Match Report
2023-10-08	16:00	Süper Lig	Matchweek 8	Sun	Home	W	2	0	İstanbulspor	65		Vincent Aboubakar	4-2-3-1	4-4-2	Tugay Kaan Numanoğlu	Match Report
2023-10-21	19:00	Süper Lig	Matchweek 9	Sat	Away	L	1	2	Galatasaray	49	50,429	Mert Günok	4-3-3	4-2-3-1	Halil Umut Meler	Match Report
2023-10-26	21:00 (22:00)	Conf Lg	Group stage	Thu	Away	L	1	3	no Bodø/Glimt		7,115	Necip Uysal	4-4-1-1	4-2-3-1	Nenad Minaković	Match Report
2023-10-30	20:00	Süper Lig	Matchweek 10	Mon	Home	W	2	0	Gaziantep FK	64		Necip Uysal	4-2-3-1	5-3-2	Mete Kalkavan	Match Report
2023-11-05	19:00	Süper Lig	Matchweek 11	Sun	Away	L	2	3	Antalyaspor	65		Necip Uysal	4-2-3-1	4-2-3-1	Yaşar Kemal Uğurlu	Match Report
2023-11-09	20:45	Conf Lg	Group stage	Thu	Home	L	1	2	no Bodø/Glimt		17,314	Necip Uysal	4-2-3-1	4-1-4-1	Sander van der Eijk	Match Report
2023-11-12	16:00	Süper Lig	Matchweek 12	Sun	Home	W	1	0	Başakşehir	49		Necip Uysal	4-3-3	4-2-3-1	Arda Kardeşler	Match Report
2023-11-26	16:00	Süper Lig	Matchweek 13	Sun	Away	W	2	1	Samsunspor	62		Necip Uysal	4-1-4-1	4-2-3-1	Abdulkadir Bitigen	Match Report
2023-11-30	20:45	Conf Lg	Group stage	Thu	Home	L	0	5	be Club Brugge		13,021	Cenk Tosun	4-2-3-1	4-2-3-1	David Smajc	Match Report
2023-12-03	19:00	Süper Lig	Matchweek 14	Sun	Away	D	1	1	Ankaragücü	60		Necip Uysal	4-1-4-1	4-2-3-1	Zorbay Küçük	Match Report
Date	Time	Comp	Round	Day	Venue	Result	GF	GA	Opponent	Poss	Attendance	Captain	Formation	Opp Formation	Referee	Match Report	Notes
2023-12-09	19:00	Süper Lig	Matchweek 15	Sat	Home	L	1	3	Fenerbahçe	46		Necip Uysal	4-1-4-1	4-2-3-1	Atilla Karaoğlan	Match Report
2023-12-14	21:00 (23:00)	Conf Lg	Group stage	Thu	Away	W	2	0	ch Lugano		5,476	Cenk Tosun	4-3-3	4-3-3	Paweł Raczkowski	Match Report
2023-12-21	20:00	Süper Lig	Matchweek 17	Thu	Home	L	1	3	Alanyaspor	59		Cenk Tosun	4-1-4-1	4-2-3-1	Umit Öztürk	Match Report
2023-12-25	20:00	Süper Lig	Matchweek 18	Mon	Away	W	2	1	Hatayspor	46		Mert Günok	4-1-4-1	4-4-2	Kadir Sağlam	Match Report
2024-01-05	20:00	Süper Lig	Matchweek 19	Fri	Home	L	1	3	Kasımpaşa	56		Necip Uysal	4-1-4-1	4-2-3-1	Arda Kardeşler	Match Report
2024-01-09	20:00	Süper Lig	Matchweek 16	Tue	Away	W	4	0	Rizespor	46		Cenk Tosun	4-1-4-1	4-2-3-1	Abdulkadir Bitigen	Match Report
2024-01-13	19:00	Süper Lig	Matchweek 20	Sat	Home	W	3	0	Fatih Karagümrük	44		Cenk Tosun	4-1-4-1	4-2-3-1	Mert Güzenge	Match Report
2024-01-20	19:00	Süper Lig	Matchweek 21	Sat	Away	L	0	4	Pendikspor	65		Cenk Tosun	4-1-4-1	4-2-3-1	Tugay Kaan Numanoğlu	Match Report
2024-01-23	20:00	Süper Lig	Matchweek 22	Tue	Home	D	0	0	Adana Demirspor	57		Necip Uysal	4-3-3	4-2-3-1	Atilla Karaoğlan	Match Report
2024-01-28	16:00	Süper Lig	Matchweek 23	Sun	Away	L	0	1	Sivasspor	71		Necip Uysal	4-3-3	4-1-4-1	Ali Şansalan	Match Report
2024-02-04	19:00	Süper Lig	Matchweek 24	Sun	Home	W	2	0	Trabzonspor	45		Necip Uysal	4-2-3-1	4-2-3-1	Arda Kardeşler	Match Report
2024-02-12	20:00	Süper Lig	Matchweek 25	Mon	Away	D	0	0	Kayserispor	46		Necip Uysal	4-2-3-1	4-1-4-1	Kadir Sağlam	Match Report
2024-02-19	20:00	Süper Lig	Matchweek 26	Mon	Home	W	2	0	Konyaspor	47		Necip Uysal	4-2-3-1	4-2-3-1	Tugay Kaan Numanoğlu	Match Report
2024-02-25	19:00	Süper Lig	Matchweek 27	Sun	Away	W	2	0	İstanbulspor	51		Necip Uysal	4-2-3-1	3-4-3	Atilla Karaoğlan	Match Report
2024-03-03	19:00	Süper Lig	Matchweek 28	Sun	Home	L	0	1	Galatasaray	48		Necip Uysal	4-2-3-1	4-2-3-1	Halil Umut Meler	Match Report
2024-03-11	20:30	Süper Lig	Matchweek 29	Mon	Away	L	0	2	Gaziantep FK	59		Necip Uysal	4-1-4-1	4-1-4-1	Arda Kardeşler	Match Report
2024-03-16	20:30	Süper Lig	Matchweek 30	Sat	Home	L	1	2	Antalyaspor	51		Cenk Tosun	4-2-3-1	4-1-4-1	Burak Pakkan	Match Report
2024-04-04	20:30	Süper Lig	Matchweek 31	Thu	Away	D	1	1	Başakşehir	30		Necip Uysal	4-1-4-1	4-3-3	Zorbay Küçük	Match Report
2024-04-13	19:00	Süper Lig	Matchweek 32	Sat	Home	D	1	1	Samsunspor	52		Necip Uysal	4-1-4-1	4-2-3-1	Direnç Tonusluoğlu	Match Report
2024-04-19	20:00	Süper Lig	Matchweek 33	Fri	Home	W	2	0	Ankaragücü	44		Necip Uysal	4-2-3-1	4-2-3-1	Bahattin Şimşek	Match Report
2024-04-27	19:00	Süper Lig	Matchweek 34	Sat	Away	L	1	2	Fenerbahçe	34		Necip Uysal	4-2-3-1	4-2-3-1	Volkan Bayarslan	Match Report
2024-05-03	20:00	Süper Lig	Matchweek 35	Fri	Home	W	3	2	Rizespor	60		Necip Uysal	4-1-4-1	4-2-3-1	Ali Şansalan	Match Report
2024-05-12	19:00	Süper Lig	Matchweek 36	Sun	Away	D	1	1	Alanyaspor	45		Necip Uysal	4-2-3-1	3-4-1-2	Tugay Kaan Numanoğlu	Match Report
2024-05-18	19:00	Süper Lig	Matchweek 37	Sat	Home	D	2	2	Hatayspor	69		Cenk Tosun	4-2-3-1	4-1-4-1	Cihan Aydın	Match Report
2024-05-26	19:00	Süper Lig	Matchweek 38	Sun	Away	L	1	2	Kasımpaşa	55		Cenk Tosun	4-3-3	4-1-4-1	Bahattin Şimşek	Match Report	""","Beşiktaş")

In [218]:
add_directly_to_data("""Date	Time	Round	Day	Venue	Result	GF	GA	Opponent	Poss	Attendance	Captain	Formation	Opp Formation	Referee	Match Report	Notes
2022-08-06	21:45	Matchweek 1	Sat	Home	W	1	0	Kayserispor	55		Necip Uysal	4-2-3-1	4-2-3-1	Umit Öztürk	Match Report
2022-08-14	21:45	Matchweek 2	Sun	Away	D	3	3	Alanyaspor	22		Rachid Ghezzal	4-1-4-1	3-4-3	Yasin Kol	Match Report
2022-08-21	21:45	Matchweek 3	Sun	Home	W	4	1	Fatih Karagümrük	47		Necip Uysal	4-1-4-1	4-2-3-1	Arda Kardeşler	Match Report
2022-08-29	21:45	Matchweek 4	Mon	Home	W	3	1	Sivasspor	59		Necip Uysal	4-2-3-1	4-1-4-1	Volkan Bayarslan	Match Report
2022-09-04	21:45	Matchweek 5	Sun	Away	W	3	2	Ankaragücü	46		Necip Uysal	4-2-3-1	4-2-3-1	Mete Kalkavan	Match Report
2022-09-12	19:00	Matchweek 6	Mon	Home	L	0	1	Başakşehir	49		Necip Uysal	4-3-3	4-3-3	Halil Umut Meler	Match Report
2022-09-17	20:00	Matchweek 7	Sat	Away	D	2	2	İstanbulspor	51		Josef de Souza	4-3-3	3-4-3	Zorbay Küçük	Match Report
2022-10-02	20:00	Matchweek 8	Sun	Home	D	0	0	Fenerbahçe	48		Josef de Souza	4-2-3-1	3-4-1-2	Volkan Bayarslan	Match Report
2022-10-08	20:00	Matchweek 9	Sat	Away	W	1	0	Giresunspor	47		Josef de Souza	4-2-3-1	4-2-3-1	Yaşar Kemal Uğurlu	Match Report
2022-10-16	20:00	Matchweek 10	Sun	Home	D	2	2	Trabzonspor	59		Josef de Souza	4-2-3-1	4-3-3	Ali Şansalan	Match Report
2022-10-24	20:00	Matchweek 11	Mon	Away	L	1	2	Hatayspor	58		Josef de Souza	4-2-3-1	4-3-3	Kadir Sağlam	Match Report
2022-10-30	17:00	Matchweek 12	Sun	Home	W	5	2	Ümraniyespor	48		Josef de Souza	4-2-3-1	4-1-4-1	Tugay Kaan Numanoğlu	Match Report
2022-11-05	20:00	Matchweek 13	Sat	Away	L	1	2	Galatasaray	46		Josef de Souza	4-3-3	4-2-3-1	Halil Umut Meler	Match Report
2022-12-25	16:00	Matchweek 15	Sun	Away	D	1	1	Gaziantep FK	65		Atiba Hutchinson	4-2-3-1	4-2-3-1	Atilla Karaoğlan	Match Report
2022-12-29	20:00	Matchweek 16	Thu	Home	W	1	0	Adana Demirspor	57		Josef de Souza	4-2-3-1	4-4-2	Ali Şansalan	Match Report
2023-01-07	19:00	Matchweek 18	Sat	Home	W	2	1	Kasımpaşa	55		Cenk Tosun	4-4-2	5-3-2	Zorbay Küçük	Match Report
2023-01-14	16:00	Matchweek 19	Sat	Away	W	2	1	Konyaspor	59		Josef de Souza	4-2-3-1	4-4-2	Abdulkadir Bitigen	Match Report
2023-01-22	16:00	Matchweek 20	Sun	Away	W	2	0	Kayserispor	38		Cenk Tosun	4-2-3-1	4-1-4-1	Mete Kalkavan	Match Report
2023-01-27	20:00	Matchweek 21	Fri	Home	W	3	0	Alanyaspor	56		Cenk Tosun	4-2-3-1	3-4-3	Yaşar Kemal Uğurlu	Match Report
2023-01-31	20:00	Matchweek 22	Tue	Away	D	1	1	Fatih Karagümrük	57		Cenk Tosun	4-2-3-1	4-3-3	Ali Şansalan	Match Report
2023-02-04	16:00	Matchweek 23	Sat	Away	L	0	1	Sivasspor	60		Necip Uysal	4-2-3-1	4-3-3	Zorbay Küçük	Match Report
2023-02-26	19:00	Matchweek 14	Sun	Home	D	0	0	Antalyaspor	61		Cenk Tosun	4-2-3-1	4-2-4	Abdulkadir Bitigen	Match Report
2023-03-05	19:00	Matchweek 24	Sun	Home	W	2	1	Ankaragücü	51		Cenk Tosun	4-1-4-1	4-2-3-1	Volkan Bayarslan	Match Report
2023-03-12	19:00	Matchweek 25	Sun	Away	W	2	0	Başakşehir	51		Cenk Tosun	4-1-4-1	4-3-3	Mete Kalkavan	Match Report
2023-03-18	19:00	Matchweek 26	Sat	Home	W	3	1	İstanbulspor	67		Cenk Tosun	4-1-4-1	4-4-1-1	Ali Şansalan	Match Report
Date	Time	Round	Day	Venue	Result	GF	GA	Opponent	Poss	Attendance	Captain	Formation	Opp Formation	Referee	Match Report	Notes
2023-04-02	20:30	Matchweek 27	Sun	Away	W	4	2	Fenerbahçe	35		Cenk Tosun	4-1-4-1	3-5-2	Halil Umut Meler	Match Report
2023-04-09	20:30	Matchweek 28	Sun	Home	W	3	1	Giresunspor	54		Cenk Tosun	4-2-3-1	4-1-4-1	Atilla Karaoğlan	Match Report
2023-04-16	20:30	Matchweek 29	Sun	Away	D	0	0	Trabzonspor	54		Rachid Ghezzal	4-2-3-1	4-1-4-1	Volkan Bayarslan	Match Report
2023-04-19		Matchweek 30	Wed	Home	W	3	0	Hatayspor							Match Report	Match awarded to Beşiktaş
2023-04-22	19:00	Matchweek 31	Sat	Away	W	2	0	Ümraniyespor	59		Cenk Tosun	4-2-3-1	4-3-3	Cihan Aydın	Match Report
2023-04-30	19:00	Matchweek 32	Sun	Home	W	3	1	Galatasaray	45		Cenk Tosun	4-2-3-1	4-2-3-1	Ali Şansalan	Match Report
2023-05-06	19:00	Matchweek 33	Sat	Away	W	3	1	Antalyaspor	68		Cenk Tosun	4-2-3-1	4-2-3-1	Mete Kalkavan	Match Report
2023-05-17		Matchweek 34	Wed	Home	W	3	0	Gaziantep FK							Match Report	Match awarded to Beşiktaş
2023-05-21	19:00	Matchweek 35	Sun	Away	W	4	1	Adana Demirspor	48		Cenk Tosun	4-3-3	4-4-2	Mustafa Kürşad Filiz	Match Report
2023-06-03	19:00	Matchweek 37	Sat	Away	W	5	2	Kasımpaşa	60		Necip Uysal	4-2-3-1	4-2-3-1	Volkan Bayarslan	Match Report
2023-06-07	20:00	Matchweek 38	Wed	Home	D	3	3	Konyaspor	60		Cenk Tosun	4-3-3	4-2-3-1	Umit Öztürk	Match Report	""","Beşiktaş")

In [219]:
add_directly_to_data("""Date	Time	Comp	Round	Day	Venue	Result	GF	GA	Opponent	Poss	Attendance	Captain	Formation	Opp Formation	Referee	Match Report	Notes
2024-07-23	20:30 (21:30)	Champions Lg	Second qualifying round	Tue	Away	W	4	3	ch Lugano		6,009	Edin Džeko	4-3-3	4-3-3	Tiago Martins	Match Report	Leg 1 of 2
2024-07-30	20:00	Champions Lg	Second qualifying round	Tue	Home	W	2	1	ch Lugano		37,281	Edin Džeko	4-3-3	4-3-3	Robert Jones	Match Report	Leg 2 of 2; Fenerbahçe won
2024-08-06	20:30 (21:30)	Champions Lg	Third qualifying round	Tue	Away	L	1	2	fr Lille		12,908	Edin Džeko	4-3-3	4-4-1-1	István Kovács	Match Report	Leg 1 of 2
2024-08-10	21:45	Süper Lig	Matchweek 1	Sat	Home	W	1	0	Adana Demirspor	53		Edin Džeko	4-2-3-1	5-4-1	Atilla Karaoğlan	Match Report
2024-08-13	20:00	Champions Lg	Third qualifying round	Tue	Home	D	1	1	fr Lille		40,474	Edin Džeko	4-3-3	4-3-1-2	Jose Maria Sánchez	Match Report	Leg 2 of 2; Lille won; Required Extra Time
2024-08-17	21:45	Süper Lig	Matchweek 2	Sat	Away	D	2	2	Göztepe	55		Edin Džeko	4-2-2-2	5-3-2	Kadir Sağlam	Match Report
2024-08-25	21:45	Süper Lig	Matchweek 3	Sun	Away	W	5	0	Rizespor	52		Edin Džeko	4-2-3-1	4-2-3-1	Turgut Doman	Match Report
2024-08-30	21:00	Süper Lig	Matchweek 4	Fri	Home	W	3	0	Alanyaspor	49		Edin Džeko	4-2-3-1	4-2-3-1	Mehmet Türkmen	Match Report
2024-09-15	17:00	Süper Lig	Matchweek 5	Sun	Away	W	2	0	Kasımpaşa	42		Dušan Tadić	4-2-3-1	5-3-2	Halil Umut Meler	Match Report
2024-09-21	20:00	Süper Lig	Matchweek 6	Sat	Home	L	1	3	Galatasaray	56		Edin Džeko	4-2-3-1	4-2-3-1	Atilla Karaoğlan	Match Report
2024-09-26	19:45	Europa Lg	League phase	Thu	Home	W	2	1	be Union SG	60	31,760	İrfan Kahveci	4-3-3	3-5-2	Benoît Bastien	Match Report
2024-09-29	19:00	Süper Lig	Matchweek 7	Sun	Away	W	2	0	Antalyaspor	49		Edin Džeko	4-2-3-1	4-3-3	Cihan Aydın	Match Report
2024-10-03	21:00 (22:00)	Europa Lg	League phase	Thu	Away	D	1	1	nl Twente	48	29,000	Dušan Tadić	4-2-3-1	4-2-3-1	Chris Kavanagh	Match Report
2024-10-20	19:00	Süper Lig	Matchweek 9	Sun	Away	D	2	2	Samsunspor	44		Dušan Tadić	4-1-4-1	4-2-3-1	Muhammetali Metoğlu	Match Report
2024-10-24	22:00	Europa Lg	League phase	Thu	Home	D	1	1	eng Manchester Utd	52	41,443	Dušan Tadić	4-3-3	4-2-3-1	Clément Turpin	Match Report
2024-10-27	19:00	Süper Lig	Matchweek 10	Sun	Home	W	2	0	BB Bodrumspor	66		Edin Džeko	4-4-2	4-2-3-1	Mehmet Türkmen	Match Report
2024-11-03	19:00	Süper Lig	Matchweek 11	Sun	Away	W	3	2	Trabzonspor	52		Dušan Tadić	4-2-3-1	4-2-3-1	Oğuzhan Çakir	Match Report
2024-11-07	21:00 (23:00)	Europa Lg	League phase	Thu	Away	L	1	3	nl AZ Alkmaar	53	15,383	Edin Džeko	4-3-3	4-3-3	John Beaton	Match Report
2024-11-10	19:00	Süper Lig	Matchweek 12	Sun	Home	W	4	0	Sivasspor	55		Dušan Tadić	4-2-3-1	4-1-4-1	Kadir Sağlam	Match Report
2024-11-23	16:00	Süper Lig	Matchweek 13	Sat	Away	W	6	2	Kayserispor	51		Dušan Tadić	4-2-3-1	5-3-2	Direnç Tonusluoğlu	Match Report
2024-11-28	21:00 (23:00)	Europa Lg	League phase	Thu	Away	W	2	1	cz Slavia Prague	47	19,308	Edin Džeko	4-3-3	3-4-3	Sascha Stegemann	Match Report
2024-12-02	20:00	Süper Lig	Matchweek 14	Mon	Home	W	3	1	Gaziantep FK	55		Dušan Tadić	4-2-3-1	3-1-4-2	Alper Akarsu	Match Report
2024-12-07	19:00	Süper Lig	Matchweek 15	Sat	Away	L	0	1	Beşiktaş	51		Edin Džeko	4-2-3-1	4-2-3-1	Mehmet Türkmen	Match Report
2024-12-11	18:30	Europa Lg	League phase	Wed	Home	L	0	2	es Athletic Club	53	35,263	Dušan Tadić	4-2-3-1	4-2-3-1	Irfan Peljto	Match Report
2024-12-15	19:00	Süper Lig	Matchweek 16	Sun	Home	W	3	1	Başakşehir	63		Edin Džeko	4-2-3-1	4-1-4-1	Ozan Ergün	Match Report
Date	Time	Comp	Round	Day	Venue	Result	GF	GA	Opponent	Poss	Attendance	Captain	Formation	Opp Formation	Referee	Match Report	Notes
2024-12-20	20:00	Süper Lig	Matchweek 17	Fri	Away	D	1	1	Eyüpspor	58		Dušan Tadić	4-2-3-1	4-1-4-1	Emre Kargın	Match Report
2025-01-05	19:00	Süper Lig	Matchweek 18	Sun	Home	W	2	1	Hatayspor	62		Edin Džeko	3-4-1-2	4-2-3-1	Oğuzhan Çakir	Match Report
2025-01-13	20:00	Süper Lig	Matchweek 19	Mon	Away	W	3	2	Konyaspor	45		Edin Džeko	3-4-1-2	4-1-4-1	Direnç Tonusluoğlu	Match Report
2025-01-19	19:00	Süper Lig	Matchweek 20	Sun	Away	W	4	0	Adana Demirspor	59		Edin Džeko	3-1-4-2	5-4-1	Halil Umut Meler	Match Report
2025-01-23	20:45	Europa Lg	League phase	Thu	Home	D	0	0	fr Lyon	44	36,722	Dušan Tadić	3-4-3	4-2-3-1	Simone Sozza	Match Report
2025-01-26	19:00	Süper Lig	Matchweek 21	Sun	Home	W	3	2	Göztepe	59		Edin Džeko	3-4-1-2	3-4-1-2	Atilla Karaoğlan	Match Report
2025-01-30	21:00 (23:00)	Europa Lg	League phase	Thu	Away	D	2	2	dk Midtjylland	46	9,693	Edin Džeko	3-4-1-2	4-3-3	Luís Godinho	Match Report
2025-02-02	19:00	Süper Lig	Matchweek 22	Sun	Home	W	3	2	Rizespor	67		Edin Džeko	3-4-1-2	4-2-3-1	Burak Pakkan	Match Report
2025-02-09	16:00	Süper Lig	Matchweek 23	Sun	Away	W	2	0	Alanyaspor	47		Edin Džeko	3-4-1-2	4-2-3-1	Cihan Aydın	Match Report
2025-02-13	20:45	Europa Lg	Knockout phase play-offs	Thu	Home	W	3	0	be Anderlecht	48	34,517	Edin Džeko	3-4-1-2	4-2-3-1	Nikola Dabanović	Match Report	Leg 1 of 2
2025-02-16	19:00	Süper Lig	Matchweek 24	Sun	Home	W	3	1	Kasımpaşa	45		Mert Yandaş	3-4-1-2	3-5-1-1	Ali Şansalan	Match Report
2025-02-20	21:00 (23:00)	Europa Lg	Knockout phase play-offs	Thu	Away	D	2	2	be Anderlecht	42	13,724	Edin Džeko	3-4-1-2	3-4-3	Sandro Schärer	Match Report	Leg 2 of 2; Fenerbahçe won
2025-02-24	20:00	Süper Lig	Matchweek 25	Mon	Away	D	0	0	Galatasaray	40		Edin Džeko	3-4-1-2	4-2-3-1	Slavko Vinčič	Match Report
2025-03-02	20:30	Süper Lig	Matchweek 26	Sun	Home	W	3	0	Antalyaspor	49		Edin Džeko	3-4-1-2	4-3-3	Mehmet Türkmen	Match Report
2025-03-06	20:45	Europa Lg	Round of 16	Thu	Home	L	1	3	sct Rangers	64	42,090	Edin Džeko	3-4-1-2	3-4-3	Alejandro Hernández	Match Report	Leg 1 of 2
2025-03-13	20:00 (23:00)	Europa Lg	Round of 16	Thu	Away	W	2 (2)	0 (3)	sct Rangers	60	50,061	Fred	3-4-3	3-4-3	Espen Eskås	Match Report	Leg 2 of 2; Rangers won; Rangers won on penalty kicks following extra time
2025-03-16	20:30	Süper Lig	Matchweek 28	Sun	Home	D	0	0	Samsunspor	65		Edin Džeko	3-4-1-2	4-2-3-1	Cihan Aydın	Match Report
2025-03-28	20:30	Süper Lig	Matchweek 29	Fri	Away	W	4	2	BB Bodrumspor	71		İrfan Kahveci	3-4-1-2	3-5-1-1	Atilla Karaoğlan	Match Report
2025-04-06	19:00	Süper Lig	Matchweek 30	Sun	Home	W	4	1	Trabzonspor	51		Edin Džeko	4-2-3-1	5-4-1	Mehmet Türkmen	Match Report
2025-04-13	19:00	Süper Lig	Matchweek 31	Sun	Away	W	3	1	Sivasspor	55		Edin Džeko	3-4-3	4-3-3	Yasin Kol	Match Report
2025-04-20	19:00	Süper Lig	Matchweek 32	Sun	Home	D	3	3	Kayserispor	72		Dušan Tadić	4-2-3-1	5-4-1	Oğuzhan Çakir	Match Report
2025-04-26	19:00	Süper Lig	Matchweek 33	Sat	Away	W	3	1	Gaziantep FK	60		Edin Džeko	3-4-1-2	3-1-4-2	Direnç Tonusluoğlu	Match Report
2025-05-04	19:00	Süper Lig	Matchweek 34	Sun	Home	L	0	1	Beşiktaş	62		Fred	4-4-2	4-2-3-1	Yasin Kol	Match Report
2025-05-09	20:00	Süper Lig	Matchweek 35	Fri	Away	W	4	1	Başakşehir	50		Edin Džeko	4-3-3	4-1-4-1	Burak Pakkan	Match Report
2025-05-18	19:00	Süper Lig	Matchweek 36	Sun	Home	W	2	1	Eyüpspor	39		Dušan Tadić	3-4-1-2	4-1-4-1	Ozan Ergün	Match Report
2025-05-26	20:00	Süper Lig	Matchweek 37	Mon	Away	L	2	4	Hatayspor	47		Edin Džeko	3-1-4-2	5-3-2	Yiğit Arslan	Match Report
2025-05-31	16:00	Süper Lig	Matchweek 38	Sat	Home	W	2	1	Konyaspor	61		İrfan Kahveci	4-2-3-1	3-4-3	Oğuzhan Çakir	Match Report	""","Fenerbahçe")

In [220]:
add_directly_to_data("""Date	Time	Comp	Round	Day	Venue	Result	GF	GA	Opponent	Poss	Attendance	Captain	Formation	Opp Formation	Referee	Match Report	Notes
2023-07-26	21:00	Conf Lg	Second qualifying round	Wed	Home	W	5	0	md FC Zimbru Chișinău		26,526	Edin Džeko	4-3-3	5-3-2	Nenad Minaković	Match Report	Leg 1 of 2
2023-08-01	20:30	Conf Lg	Second qualifying round	Tue	Away	W	4	0	md FC Zimbru Chișinău		7,569	Dušan Tadić	4-3-3	5-3-2	Robert Harvey	Match Report	Leg 2 of 2; Fenerbahçe won
2023-08-10	21:00	Conf Lg	Third qualifying round	Thu	Home	W	3	1	si NK Maribor		33,015	Edin Džeko	4-3-3	3-4-3	Robert Jones	Match Report	Leg 1 of 2
2023-08-13	21:45	Süper Lig	Matchweek 1	Sun	Home	W	2	1	Gaziantep FK	70		Edin Džeko	4-1-4-1	4-2-3-1	Abdulkadir Bitigen	Match Report
2023-08-17	20:15 (21:15)	Conf Lg	Third qualifying round	Thu	Away	W	3	0	si NK Maribor		10,100	Miha Zajc	4-3-3	4-2-3-1	Fábio Veríssimo	Match Report	Leg 2 of 2; Fenerbahçe won
2023-08-21	21:00	Süper Lig	Matchweek 2	Mon	Away	W	2	0	Samsunspor	60		Edin Džeko	4-1-4-1	3-4-3	Volkan Bayarslan	Match Report
2023-08-24	20:00	Conf Lg	Play-off round	Thu	Home	W	5	1	nl Twente		38,961	Dušan Tadić	4-3-3	4-2-3-1	Andy Madley	Match Report	Leg 1 of 2
2023-08-31	19:00 (20:00)	Conf Lg	Play-off round	Thu	Away	W	1	0	nl Twente		22,500	Edin Džeko	4-3-3	4-2-3-1	Fabio Maresca	Match Report	Leg 2 of 2; Fenerbahçe won
2023-09-03	21:45	Süper Lig	Matchweek 4	Sun	Away	W	1	0	Ankaragücü	49		Edin Džeko	4-2-3-1	4-2-3-1	Arda Kardeşler	Match Report
2023-09-17	17:00	Süper Lig	Matchweek 5	Sun	Home	W	3	2	Antalyaspor	60		Edin Džeko	4-2-3-1	4-2-3-1	Halil Umut Meler	Match Report
2023-09-21	19:45	Conf Lg	Group stage	Thu	Home	W	3	1	dk Nordsjælland		36,130	Mert Yandaş	4-2-3-1	4-3-3	Gergő Bogár	Match Report
2023-09-24	20:00	Süper Lig	Matchweek 6	Sun	Away	W	1	0	Alanyaspor	57		Edin Džeko	4-2-3-1	4-4-1-1	Atilla Karaoğlan	Match Report
2023-09-28	20:00	Süper Lig	Matchweek 3	Thu	Home	W	4	0	Başakşehir	58		Edin Džeko	4-2-3-1	4-4-2	Zorbay Küçük	Match Report
2023-10-01	16:00	Süper Lig	Matchweek 7	Sun	Home	W	5	0	Rizespor	66		Edin Džeko	4-2-3-1	4-2-3-1	Cihan Aydın	Match Report
2023-10-05	21:00 (22:00)	Conf Lg	Group stage	Thu	Away	W	2	1	sk Spartak Trnava		17,528	Mert Yandaş	4-3-3	3-4-3	Sascha Stegemann	Match Report
2023-10-08	19:00	Süper Lig	Matchweek 8	Sun	Away	W	2	0	Kasımpaşa	62		Edin Džeko	4-2-3-1	4-1-4-1	Mete Kalkavan	Match Report
2023-10-22	19:00	Süper Lig	Matchweek 9	Sun	Home	W	4	2	Hatayspor	55		Edin Džeko	4-2-3-1	3-4-3	Volkan Bayarslan	Match Report
2023-10-26	19:45	Conf Lg	Group stage	Thu	Home	W	3	1	bg Ludogorets Razgrad		35,595	Dušan Tadić	4-3-3	4-2-3-1	Guillermo Cuadra	Match Report
2023-10-29	19:00	Süper Lig	Matchweek 10	Sun	Away	W	5	0	Pendikspor	62		Edin Džeko	4-2-3-1	4-4-2	Abdulkadir Bitigen	Match Report
2023-11-04	19:00	Süper Lig	Matchweek 11	Sat	Home	L	2	3	Trabzonspor	65		Edin Džeko	4-2-3-1	4-2-3-1	Zorbay Küçük	Match Report
2023-11-09	22:00 (23:00)	Conf Lg	Group stage	Thu	Away	L	0	2	bg Ludogorets Razgrad		10,363	Dušan Tadić	4-3-3	4-3-3	Ondřej Berka	Match Report
2023-11-12	19:00	Süper Lig	Matchweek 12	Sun	Away	D	0	0	Adana Demirspor	43		Edin Džeko	4-2-3-1	4-2-3-1	Halil Umut Meler	Match Report
2023-11-26	19:00	Süper Lig	Matchweek 13	Sun	Home	W	2	1	Fatih Karagümrük	54		Edin Džeko	4-2-3-1	4-1-4-1	Arda Kardeşler	Match Report
2023-11-30	21:00 (23:00)	Conf Lg	Group stage	Thu	Away	L	1	6	dk Nordsjælland		7,323	Dušan Tadić	4-2-3-1	4-3-3	Jérémie Pignard	Match Report
2023-12-04	20:00	Süper Lig	Matchweek 14	Mon	Home	W	4	1	Sivasspor	49		Edin Džeko	4-2-3-1	4-1-4-1	Bahattin Şimşek	Match Report
Date	Time	Comp	Round	Day	Venue	Result	GF	GA	Opponent	Poss	Attendance	Captain	Formation	Opp Formation	Referee	Match Report	Notes
2023-12-09	19:00	Süper Lig	Matchweek 15	Sat	Away	W	3	1	Beşiktaş	54		Edin Džeko	4-2-3-1	4-1-4-1	Atilla Karaoğlan	Match Report
2023-12-14	20:45	Conf Lg	Group stage	Thu	Home	W	4	0	sk Spartak Trnava		33,936	Edin Džeko	4-2-3-1	4-3-3	Aleksandar Stavrev	Match Report
2023-12-20	17:00	Süper Lig	Matchweek 17	Wed	Away	W	4	3	Kayserispor	49		Dušan Tadić	4-2-3-1	4-2-3-1	Zorbay Küçük	Match Report
2023-12-24	19:00	Süper Lig	Matchweek 18	Sun	Home	D	0	0	Galatasaray	46		Edin Džeko	4-2-3-1	4-2-3-1	Arda Kardeşler	Match Report
2024-01-07	16:00	Süper Lig	Matchweek 19	Sun	Away	W	5	1	İstanbulspor	54		Edin Džeko	4-2-3-1	4-3-3	Volkan Bayarslan	Match Report
2024-01-10	20:00	Süper Lig	Matchweek 16	Wed	Home	W	7	1	Konyaspor	67		Edin Džeko	4-2-3-1	4-2-3-1	Kadir Sağlam	Match Report
2024-01-14	19:00	Süper Lig	Matchweek 20	Sun	Away	W	1	0	Gaziantep FK	64		Edin Džeko	4-2-3-1	5-4-1	Tugay Kaan Numanoğlu	Match Report
2024-01-21	16:00	Süper Lig	Matchweek 21	Sun	Home	D	1	1	Samsunspor	66		Edin Džeko	4-2-3-1	4-1-4-1	Bahattin Şimşek	Match Report
2024-01-24	20:00	Süper Lig	Matchweek 22	Wed	Away	W	1	0	Başakşehir	67		Edin Džeko	4-2-3-1	4-1-4-1	Abdulkadir Bitigen	Match Report
2024-01-28	19:00	Süper Lig	Matchweek 23	Sun	Home	W	2	1	Ankaragücü	63		Edin Džeko	4-2-3-1	4-2-3-1	Cihan Aydın	Match Report
2024-02-03	19:00	Süper Lig	Matchweek 24	Sat	Away	W	2	0	Antalyaspor	46		Edin Džeko	4-2-3-1	4-1-4-1	Volkan Bayarslan	Match Report
2024-02-11	19:00	Süper Lig	Matchweek 25	Sun	Home	D	2	2	Alanyaspor	64		Edin Džeko	4-2-3-1	4-2-3-1	Çağdaş Altay	Match Report
2024-02-17	19:00	Süper Lig	Matchweek 26	Sat	Away	W	3	1	Rizespor	45		Edin Džeko	4-2-3-1	4-2-3-1	Arda Kardeşler	Match Report
2024-02-24	19:00	Süper Lig	Matchweek 27	Sat	Home	W	2	1	Kasımpaşa	65		Edin Džeko	4-1-4-1	4-1-4-1	Cihan Aydın	Match Report
2024-03-02	19:00	Süper Lig	Matchweek 28	Sat	Away	W	2	0	Hatayspor	51		Edin Džeko	4-2-3-1	4-2-3-1	Atilla Karaoğlan	Match Report
2024-03-07	21:00 (23:00)	Conf Lg	Round of 16	Thu	Away	W	3	0	be Union SG	49	13,522	Dušan Tadić	4-2-3-1	3-5-2	João Pinheiro	Match Report	Leg 1 of 2
2024-03-10	19:00	Süper Lig	Matchweek 29	Sun	Home	W	4	1	Pendikspor	67		Edin Džeko	4-2-3-1	4-2-3-1	Kadir Sağlam	Match Report
2024-03-14	20:45	Conf Lg	Round of 16	Thu	Home	L	0	1	be Union SG	49	35,605	İrfan Kahveci	4-2-3-1	3-5-2	Nikola Dabanović	Match Report	Leg 2 of 2; Fenerbahçe won
2024-03-17	20:30	Süper Lig	Matchweek 30	Sun	Away	W	3	2	Trabzonspor	43		Edin Džeko	4-2-3-1	4-2-3-1	Halil Umut Meler	Match Report
2024-04-03	20:30	Süper Lig	Matchweek 31	Wed	Home	W	4	2	Adana Demirspor	58		Edin Džeko	4-2-3-1	4-3-3	Arda Kardeşler	Match Report
2024-04-11	19:45	Conf Lg	Quarter-finals	Thu	Away	L	2	3	gr Olympiacos	61	32,000	Edin Džeko	4-4-2	4-2-3-1	Sandro Schärer	Match Report	Leg 1 of 2
2024-04-14	19:00	Süper Lig	Matchweek 32	Sun	Away	W	2	1	Fatih Karagümrük	55		Edin Džeko	4-2-3-1	4-2-3-1	Tugay Kaan Numanoğlu	Match Report
2024-04-18	22:00	Conf Lg	Quarter-finals	Thu	Home	W	1 (2)	0 (3)	gr Olympiacos	62	44,040	Edin Džeko	4-3-3	4-2-3-1	Tobias Stieler	Match Report	Leg 2 of 2; Olympiacos won; Olympiacos won on penalty kicks following extra time
2024-04-22	20:00	Süper Lig	Matchweek 33	Mon	Away	D	2	2	Sivasspor	64		Edin Džeko	4-2-3-1	5-4-1	Cihan Aydın	Match Report
2024-04-27	19:00	Süper Lig	Matchweek 34	Sat	Home	W	2	1	Beşiktaş	66		Dušan Tadić	4-2-3-1	4-2-3-1	Volkan Bayarslan	Match Report
2024-05-06	20:00	Süper Lig	Matchweek 35	Mon	Away	D	0	0	Konyaspor	71		Edin Džeko	4-2-3-1	5-3-2	Bahattin Şimşek	Match Report
2024-05-12	19:00	Süper Lig	Matchweek 36	Sun	Home	W	3	0	Kayserispor	50		Dušan Tadić	4-2-3-1	4-2-3-1	Direnç Tonusluoğlu	Match Report
2024-05-19	19:00	Süper Lig	Matchweek 37	Sun	Away	W	1	0	Galatasaray	31		Dušan Tadić	4-2-3-1	4-2-3-1	Arda Kardeşler	Match Report
2024-05-26	19:00	Süper Lig	Matchweek 38	Sun	Home	W	6	0	İstanbulspor	54		Edin Džeko	4-4-2	5-4-1	Volkan Bayarslan	Match Report	""","Fenerbahçe")

In [221]:
add_directly_to_data("""Date	Time	Comp	Round	Day	Venue	Result	GF	GA	Opponent	Poss	Attendance	Captain	Formation	Opp Formation	Referee	Match Report	Notes
2022-07-20	21:00	Champions Lg	Second qualifying round	Wed	Away	D	0	0	ua Dynamo Kyiv		11,603	Altay Bayındır	4-3-3	4-4-2	Glenn Nyberg	Match Report	Leg 1 of 2
2022-07-27	20:00	Champions Lg	Second qualifying round	Wed	Home	L	1	2	ua Dynamo Kyiv		40,000	Altay Bayındır	4-1-3-2	4-2-3-1	Massimiliano Irrati	Match Report	Leg 2 of 2; Dynamo Kyiv won; Required Extra Time
2022-08-04	20:00	Europa Lg	Third qualifying round	Thu	Home	W	3	0	cz Slovácko	100	35,536	Enner Valencia	4-1-3-2	4-1-4-1	João Pinheiro	Match Report	Leg 1 of 2
2022-08-08	21:45	Süper Lig	Matchweek 1	Mon	Home	D	3	3	Ümraniyespor	53		Altay Bayındır	4-4-2◆	4-1-4-1	Yaşar Kemal Uğurlu	Match Report
2022-08-11	19:00 (20:00)	Europa Lg	Third qualifying round	Thu	Away	D	1	1	cz Slovácko		6,520	Altay Bayındır	4-3-3	4-3-3	Nikola Dabanović	Match Report	Leg 2 of 2; Fenerbahçe won
2022-08-15	21:45	Süper Lig	Matchweek 2	Mon	Away	W	6	0	Kasımpaşa	72		Altay Bayındır	4-4-2	4-1-4-1	Mete Kalkavan	Match Report
2022-08-18	21:00 (22:00)	Europa Lg	Play-off round	Thu	Away	W	2	0	at Austria Wien	62	14,000	Altay Bayındır	4-4-2	4-4-2	Daniel Siebert	Match Report	Leg 1 of 2
2022-08-22	21:45	Süper Lig	Matchweek 3	Mon	Home	W	4	2	Adana Demirspor	49		Altay Bayındır	4-4-2	4-2-3-1	Abdulkadir Bitigen	Match Report
2022-08-25	20:00	Europa Lg	Play-off round	Thu	Home	W	4	1	at Austria Wien	65	35,350	Altay Bayındır	4-4-2	4-2-3-1	Jesús Gil	Match Report	Leg 2 of 2; Fenerbahçe won
2022-08-29	19:15	Süper Lig	Matchweek 4	Mon	Away	L	0	1	Konyaspor	46		Altay Bayındır	3-5-2	4-2-3-1	Arda Kardeşler	Match Report
2022-09-03	21:45	Süper Lig	Matchweek 5	Sat	Home	W	2	0	Kayserispor	53		Altay Bayındır	3-5-2	4-3-3	Ali Palabıyık	Match Report
2022-09-08	19:45	Europa Lg	Group stage	Thu	Home	W	2	1	ua Dynamo Kyiv	65	41,895	Altay Bayındır	3-4-3	4-4-2	Tamás Bognár	Match Report
2022-09-15	21:00 (22:00)	Europa Lg	Group stage	Thu	Away	D	2	2	fr Rennes	64	20,993	Altay Bayındır	4-4-2	4-3-3	Aliaksei Kulbakou	Match Report
2022-09-18	20:00	Süper Lig	Matchweek 7	Sun	Home	W	5	0	Alanyaspor	60		Altay Bayındır	4-4-2	4-1-4-1	Hüseyin Göçek	Match Report
2022-10-02	20:00	Süper Lig	Matchweek 8	Sun	Away	D	0	0	Beşiktaş	53		Altay Bayındır	3-4-1-2	4-2-3-1	Volkan Bayarslan	Match Report
2022-10-06	22:00	Europa Lg	Group stage	Thu	Home	W	2	0	cy AEK Larnaca FC	68	38,860	Altay Bayındır	4-1-4-1	4-3-3	Chris Kavanagh	Match Report
2022-10-09	20:00	Süper Lig	Matchweek 9	Sun	Home	W	5	4	Fatih Karagümrük	55		Altay Bayındır	4-1-3-2	4-3-3	Atilla Karaoğlan	Match Report
2022-10-13	19:45	Europa Lg	Group stage	Thu	Away	W	2	1	cy AEK Larnaca FC	60	6,947	Altay Bayındır	4-3-3	4-1-4-1	Paweł Raczkowski	Match Report
2022-10-17	20:00	Süper Lig	Matchweek 10	Mon	Away	W	3	0	Ankaragücü	46		Altay Bayındır	3-1-4-2	4-2-3-1	Halil Umut Meler	Match Report
2022-10-22	20:00	Süper Lig	Matchweek 11	Sat	Home	W	1	0	Başakşehir	56		Altay Bayındır	3-1-4-2	4-3-3	Zorbay Küçük	Match Report
2022-10-27	19:45	Europa Lg	Group stage	Thu	Home	D	3	3	fr Rennes	63	34,840	Altay Bayındır	3-4-3	4-4-2	Novak Simović	Match Report
2022-10-30	20:00	Süper Lig	Matchweek 12	Sun	Away	W	5	2	İstanbulspor	63		Altay Bayındır	4-4-2	4-1-4-1	Mustafa Kürşad Filiz	Match Report
2022-11-03	21:00 (23:00)	Europa Lg	Group stage	Thu	Away	W	2	0	ua Dynamo Kyiv	60	6,304	Altay Bayındır	4-3-3	4-2-3-1	Duje Strukan	Match Report
2022-11-07	20:00	Süper Lig	Matchweek 13	Mon	Home	W	1	0	Sivasspor			Altay Bayındır	4-1-3-2	4-2-3-1	Hüseyin Göçek	Match Report
2022-11-12	17:00	Süper Lig	Matchweek 14	Sat	Home	L	1	2	Giresunspor	50		Altay Bayındır	4-1-3-2	4-2-3-1	Arda Kardeşler	Match Report
Date	Time	Comp	Round	Day	Venue	Result	GF	GA	Opponent	Poss	Attendance	Captain	Formation	Opp Formation	Referee	Match Report	Notes
2022-12-24	19:00	Süper Lig	Matchweek 15	Sat	Away	L	0	2	Trabzonspor	47		Altay Bayındır	3-4-3	4-1-4-1	Halil Umut Meler	Match Report
2022-12-27	20:00	Süper Lig	Matchweek 16	Tue	Home	W	4	0	Hatayspor	66		Altay Bayındır	4-4-2	4-2-3-1	Sarper Barış Saka	Match Report
2023-01-03	20:00	Süper Lig	Matchweek 17	Tue	Away	W	2	1	Antalyaspor	64		Altay Bayındır	4-4-2	3-4-1-2	Kadir Sağlam	Match Report
2023-01-08	19:00	Süper Lig	Matchweek 18	Sun	Home	L	0	3	Galatasaray	63		Altay Bayındır	3-4-3	4-2-3-1	Halil Umut Meler	Match Report
2023-01-15	19:00	Süper Lig	Matchweek 19	Sun	Away	W	2	1	Gaziantep FK	69		Altay Bayındır	4-4-2	4-4-2	Mustafa Kürşad Filiz	Match Report
2023-01-23	20:00	Süper Lig	Matchweek 20	Mon	Away	W	2	1	Ümraniyespor	68		Altay Bayındır	4-4-2	4-3-3	Abdulkadir Bitigen	Match Report
2023-01-29	19:00	Süper Lig	Matchweek 21	Sun	Home	W	5	1	Kasımpaşa	63		Altay Bayındır	4-1-3-2	4-2-3-1	Volkan Bayarslan	Match Report
2023-02-02	20:00	Süper Lig	Matchweek 22	Thu	Away	D	1	1	Adana Demirspor	62		Altay Bayındır	4-1-3-2	4-2-4	Ali Palabıyık	Match Report
2023-02-25	19:00	Süper Lig	Matchweek 23	Sat	Home	W	4	0	Konyaspor	71		Altay Bayındır	4-1-3-2	4-2-3-1	Halil Umut Meler	Match Report
2023-03-04	19:00	Süper Lig	Matchweek 24	Sat	Away	W	2	1	Kayserispor	59		Altay Bayındır	4-4-2◆	4-1-4-1	Atilla Karaoğlan	Match Report
2023-03-09	21:00 (23:00)	Europa Lg	Round of 16	Thu	Away	L	0	2	es Sevilla	47	24,480	Altay Bayındır	4-2-2-2	5-3-2	François Letexier	Match Report	Leg 1 of 2
2023-03-16	20:45	Europa Lg	Round of 16	Thu	Home	W	1	0	es Sevilla	66	44,775	Altay Bayındır	4-4-2	5-4-1	Michael Oliver	Match Report	Leg 2 of 2; Sevilla won
2023-03-19	19:00	Süper Lig	Matchweek 26	Sun	Away	W	3	1	Alanyaspor	71		Altay Bayındır	4-4-2	4-2-3-1	Mete Kalkavan	Match Report
2023-04-02	20:30	Süper Lig	Matchweek 27	Sun	Home	L	2	4	Beşiktaş	65		Altay Bayındır	3-5-2	4-1-4-1	Halil Umut Meler	Match Report
2023-04-10	20:30	Süper Lig	Matchweek 28	Mon	Away	W	2	1	Fatih Karagümrük	64		Altay Bayındır	4-4-2	4-3-3	Abdulkadir Bitigen	Match Report
2023-04-15	20:30	Süper Lig	Matchweek 29	Sat	Home	W	2	1	Ankaragücü	69		Enner Valencia	4-4-2	5-4-1	Atilla Karaoğlan	Match Report
2023-04-19	20:30	Süper Lig	Matchweek 30	Wed	Away	W	2	1	Başakşehir	66		Enner Valencia	4-1-3-2	4-2-3-1	Suat Arslanboga	Match Report
2023-04-24	20:00	Süper Lig	Matchweek 31	Mon	Home	D	3	3	İstanbulspor	73		Mert Yandaş	4-1-3-2	4-1-4-1	Kadir Sağlam	Match Report
2023-04-29	19:00	Süper Lig	Matchweek 32	Sat	Away	W	3	1	Sivasspor	47		Enner Valencia	4-4-2	4-1-4-1	Zorbay Küçük	Match Report
2023-05-07	19:00	Süper Lig	Matchweek 33	Sun	Away	D	1	1	Giresunspor	67		Mert Yandaş	4-4-2	4-1-4-1	Bahattin Şimşek	Match Report
2023-05-18	20:00	Süper Lig	Matchweek 34	Thu	Home	W	3	1	Trabzonspor	56		Enner Valencia	4-4-2	5-3-2	Abdulkadir Bitigen	Match Report
2023-05-21		Süper Lig	Matchweek 35	Sun	Away	W	3	0	Hatayspor							Match Report	Match awarded to Fenerbahçe
2023-05-30	20:00	Süper Lig	Matchweek 36	Tue	Home	W	2	0	Antalyaspor	61		Enner Valencia	4-1-3-2	5-4-1	Zorbay Küçük	Match Report
2023-06-04	19:00	Süper Lig	Matchweek 37	Sun	Away	L	0	3	Galatasaray	48		Enner Valencia	4-1-3-2	4-2-3-1	Abdulkadir Bitigen	Match Report
2023-06-07		Süper Lig	Matchweek 38	Wed	Home	W	3	0	Gaziantep FK							Match Report	Match awarded to Fenerbahçe""","Fenerbahçe")